# 00 以480m数据清理

In [30]:
import os
import geopandas as gpd

grid_folder = r'D:\seoul\grids\lst_map\final_clean'
output_folder = r'D:\seoul\grids\lst_map\final_clean\480_based'

os.makedirs(output_folder, exist_ok=True)

for filename in os.listdir(grid_folder):
    if filename.endswith('480m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp'):
        filepath = os.path.join(grid_folder, filename)
        gdf = gpd.read_file(filepath)

        if 'lst_ratio' in gdf.columns:
            before_count = len(gdf)
            gdf_clean = gdf[gdf['lst_ratio'] >= 0.8]
            after_count = len(gdf_clean)

            if before_count != after_count:
                output_path = os.path.join(output_folder, filename)
                gdf_clean.to_file(output_path)
                print(f"✔ 清理 {filename}：原始 {before_count} 行，清理后 {after_count} 行")


✔ 清理 city2016_lst_ratio_grid_480m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp：原始 2245 行，清理后 1861 行
✔ 清理 city2023_lst_ratio_grid_480m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp：原始 2148 行，清理后 1572 行


# 00 根据480m的有效范围去裁剪120m和240m

In [32]:
import os
import geopandas as gpd

# 工作路径与数据路径
work_space = r'D:\seoul\grids\lst_map\final_clean'
year = 2016
data_file_120 = os.path.join(work_space, f'city{year}_lst_ratio_grid_120m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp')
data_file_240 = os.path.join(work_space, f'city{year}_lst_ratio_grid_240m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp')
data_file_480 = os.path.join(work_space, r'480_based', f'city{year}_lst_ratio_grid_480m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp')

output_space = os.path.join(work_space, '480_based')
os.makedirs(output_space, exist_ok=True)

def clean_file_by_within(input_file, base_file, output_file):
    gdf_input = gpd.read_file(input_file)
    gdf_base = gpd.read_file(base_file)

    # 确保投影一致
    if gdf_input.crs != gdf_base.crs:
        gdf_base = gdf_base.to_crs(gdf_input.crs)

    # 将 480m 网格合并为一个大的几何体集合
    base_union = gdf_base.unary_union
    gdf_filtered = gdf_input[gdf_input.geometry.buffer(-0.0001).within(base_union)]

    # # 判断每个小网格是否完全包含于480网格中
    # gdf_filtered = gdf_input[gdf_input.geometry.within(base_union)]

    # 保存输出
    gdf_filtered.to_file(output_file)
    print(f"✔ 完成清洗: {output_file}，保留 {len(gdf_filtered)} 个网格.")

# 输出路径定义
output_file_120 = os.path.join(output_space, os.path.basename(data_file_120))
output_file_240 = os.path.join(output_space, os.path.basename(data_file_240))

# 执行清洗
clean_file_by_within(data_file_120, data_file_480, output_file_120)
clean_file_by_within(data_file_240, data_file_480, output_file_240)


✔ 完成清洗: D:\seoul\grids\lst_map\final_clean\480_based\city2016_lst_ratio_grid_120m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp，保留 29707 个网格.
✔ 完成清洗: D:\seoul\grids\lst_map\final_clean\480_based\city2016_lst_ratio_grid_240m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp，保留 7442 个网格.


# 00 清理 120m 和 240m 的 lst_ratio <0.8

In [33]:
import os
import geopandas as gpd

grid_folder = r'D:\seoul\grids\lst_map\final_clean\480_based'
output_folder = r'D:\seoul\grids\lst_map\final_clean\480_based'

os.makedirs(output_folder, exist_ok=True)

for filename in os.listdir(grid_folder):
    if filename.endswith('bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp'):
        filepath = os.path.join(grid_folder, filename)
        gdf = gpd.read_file(filepath)

        if 'lst_ratio' in gdf.columns:
            before_count = len(gdf)
            gdf_clean = gdf[gdf['lst_ratio'] >= 0.8]
            after_count = len(gdf_clean)

            if before_count != after_count:
                output_path = os.path.join(output_folder, filename)
                gdf_clean.to_file(output_path)
                print(f"✔ 清理 {filename}：原始 {before_count} 行，清理后 {after_count} 行")


✔ 清理 city2016_lst_ratio_grid_120m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp：原始 29707 行，清理后 29283 行
✔ 清理 city2016_lst_ratio_grid_240m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp：原始 7442 行，清理后 7338 行
✔ 清理 city2023_lst_ratio_grid_120m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp：原始 25121 行，清理后 24306 行
✔ 清理 city2023_lst_ratio_grid_240m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp：原始 6288 行，清理后 6093 行


# 00 + M -> KM

In [58]:
import os
import geopandas as gpd

grid_folder = r'D:\seoul\grids\lst_map\final_clean\480_based'
output_folder = r'D:\seoul\grids\lst_map\final_clean\480_based'

os.makedirs(output_folder, exist_ok=True)

for filename in os.listdir(grid_folder):
    if filename.endswith('bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp'):
        filepath = os.path.join(grid_folder, filename)
        gdf = gpd.read_file(filepath)

        # 标记是否有修改
        modified = False

        # 转换三个距离字段：米 → 公里
        for col in ['Dist_WB', 'Dist_BP', 'Dist_MT']:
            if col in gdf.columns:
                gdf[col] = gdf[col] / 1000.0
                modified = True

        if modified:
            output_path = os.path.join(output_folder, filename)
            gdf.to_file(output_path)
            print(f"✔ 已更新 {filename}：三个距离字段已转换为公里")
        else:
            print(f"ℹ️ 跳过 {filename}：未找到指定字段")


✔ 已更新 city2016_lst_ratio_grid_120m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp：三个距离字段已转换为公里
✔ 已更新 city2016_lst_ratio_grid_240m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp：三个距离字段已转换为公里
✔ 已更新 city2016_lst_ratio_grid_480m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp：三个距离字段已转换为公里
✔ 已更新 city2023_lst_ratio_grid_120m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp：三个距离字段已转换为公里
✔ 已更新 city2023_lst_ratio_grid_240m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp：三个距离字段已转换为公里
✔ 已更新 city2023_lst_ratio_grid_480m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp：三个距离字段已转换为公里


# 00 + 集体改名
explanatory_vars = ['BCR(%)', 'BHV', 'NDVI', 'SVF', 'EV(m)',
                    'Dist_BP', 'Dist_MT', 'Dist_WB', 'WR(%)']

explanatory_vars = ['BCR', 'BHV',  'SVF', 'NDVI', 'EV', 'WR', 'Dist_P', 'Dist_M', 'Dist_W']   

In [ ]:
import os
import geopandas as gpd

grid_folder = r'D:\seoul\grids\lst_map\final_clean\480_based'
output_folder = r'D:\seoul\grids\lst_map\final_clean\480_based'

os.makedirs(output_folder, exist_ok=True)

for filename in os.listdir(grid_folder):
    if filename.endswith('bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp'):
        filepath = os.path.join(grid_folder, filename)
        gdf = gpd.read_file(filepath)

        # 改字段名
        gdf = gdf.rename(columns={
            'Dist_WB': 'Dist_W',
            'Dist_BP': 'Dist_P',
            'Dist_MT': 'Dist_M'
        })

        # 保存 shapefile
        output_path = os.path.join(output_folder, filename)
        gdf.to_file(output_path)

In [129]:
import os
import geopandas as gpd

grid_folder = r'D:\seoul\grids\lst_map\final_clean\480_based'
output_folder = r'D:\seoul\grids\lst_map\final_clean\480_based'

os.makedirs(output_folder, exist_ok=True)


for filename in os.listdir(grid_folder):
    if filename.endswith('bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp'):
        filepath = os.path.join(grid_folder, filename)
        gdf = gpd.read_file(filepath)

        # 改字段名
        gdf = gdf.rename(columns={
            'BCR(%)': 'BCR',
            'EV(m)': 'EV',
            'WR(%)': 'WR'
        })

        # 保存 shapefile
        output_path = os.path.join(output_folder, filename)
        gdf.to_file(output_path)

# 00 画一个新的区域范围图

In [38]:
import geopandas as gpd
import matplotlib.pyplot as plt
import os

# 首尔行政边界
shp_seoul = r'D:\seoul\Final_data\Admin_boundary\Seoul_boundary.shp'
gdf_seoul = gpd.read_file(shp_seoul)

# 输出文件夹
output_folder = r'D:\seoul\grids\lst_map\final_clean\480_based\figures'
os.makedirs(output_folder, exist_ok=True)
grid_sizes = [120,240,480]
for year in [2016,2023]:

    for i in grid_sizes:
        shp_path = rf'D:\seoul\grids\lst_map\final_clean\480_based\city{year}_lst_ratio_grid_{i}m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp'

        gdf = gpd.read_file(shp_path)

        # 如果两者 CRS 不一致，投影到相同坐标系
        if gdf.crs != gdf_seoul.crs:
            gdf_seoul = gdf_seoul.to_crs(gdf.crs)

        fig, ax = plt.subplots(figsize=(8, 8))
        # 再叠加边界线
        gdf_seoul.plot(ax=ax, facecolor='none', edgecolor='black', linewidth=0.8)
        # 先画格网
        gdf.plot(ax=ax, facecolor='lightgray', edgecolor='gray', linewidth=0.3)

        ax.set_title(f'Grid Size: {i}m', fontsize=14)
        ax.axis('off')

        output_path = os.path.join(output_folder, f'{year}_grid_{i}m_gray.png')
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"✅ Saved: {output_path}")

✅ Saved: D:\seoul\grids\lst_map\final_clean\480_based\figures\2016_grid_120m_gray.png
✅ Saved: D:\seoul\grids\lst_map\final_clean\480_based\figures\2016_grid_240m_gray.png
✅ Saved: D:\seoul\grids\lst_map\final_clean\480_based\figures\2016_grid_480m_gray.png
✅ Saved: D:\seoul\grids\lst_map\final_clean\480_based\figures\2023_grid_120m_gray.png
✅ Saved: D:\seoul\grids\lst_map\final_clean\480_based\figures\2023_grid_240m_gray.png
✅ Saved: D:\seoul\grids\lst_map\final_clean\480_based\figures\2023_grid_480m_gray.png


# 00 基于480m的grid打标签FID_1 方便最后比较

In [35]:
years=[2016,2023]
for year in years:
    shp_path = rf'D:\seoul\grids\lst_map\final_clean\480_based\city{year}_lst_ratio_grid_480m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp'
    file_480= gpd.read_file(shp_path)
    file_480['FID_1'] = file_480.index
    file_480.to_file(shp_path)

In [36]:
import geopandas as gpd

def assign_fid_from_larger_grid(small_grid_path, large_grid_path, fid_field='FID_1'):
    """
    将较小网格的每个单元中心点匹配到较大网格中的FID，并写入原shapefile。

    Parameters:
        small_grid_path (str): 较小网格（如120m或240m）的Shapefile路径
        large_grid_path (str): 较大网格（如480m）的Shapefile路径，需包含FID字段
        fid_field (str): 在大网格中要匹配的字段名，默认是 'FID_1'
    """

    print(f"🔍 正在处理文件：{small_grid_path}")

    # 读取两个图层
    large_gdf = gpd.read_file(large_grid_path)
    small_gdf = gpd.read_file(small_grid_path)

    # 确保投影一致
    large_gdf = large_gdf.to_crs(small_gdf.crs)

    # 计算小网格中心点
    small_gdf['centroid'] = small_gdf.geometry.centroid
    centroids = gpd.GeoDataFrame(small_gdf[['centroid']], geometry='centroid', crs=small_gdf.crs)

    # 执行空间连接：中心点 within 大网格
    joined = gpd.sjoin(centroids, large_gdf[[fid_field, 'geometry']], how='left', predicate='within')

    # 将FID字段赋值给原始数据
    small_gdf[fid_field] = joined[fid_field].values

    # 删除中间计算列
    small_gdf = small_gdf.drop(columns='centroid')

    # 保存回原始路径
    small_gdf.to_file(small_grid_path)
    print(f"✔ 完成并保存：{small_grid_path}\n")

for year in [2016,2023]:
    for grid in [120,240]:
        assign_fid_from_larger_grid(
            small_grid_path=rf'D:\seoul\grids\lst_map\final_clean\480_based\city{year}_lst_ratio_grid_{grid}m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp',
            large_grid_path=rf'D:\seoul\grids\lst_map\final_clean\480_based\city{year}_lst_ratio_grid_480m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp'
        )



🔍 正在处理文件：D:\seoul\grids\lst_map\final_clean\480_based\city2016_lst_ratio_grid_120m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp
✔ 完成并保存：D:\seoul\grids\lst_map\final_clean\480_based\city2016_lst_ratio_grid_120m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp

🔍 正在处理文件：D:\seoul\grids\lst_map\final_clean\480_based\city2016_lst_ratio_grid_240m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp
✔ 完成并保存：D:\seoul\grids\lst_map\final_clean\480_based\city2016_lst_ratio_grid_240m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp

🔍 正在处理文件：D:\seoul\grids\lst_map\final_clean\480_based\city2023_lst_ratio_grid_120m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp
✔ 完成并保存：D:\seoul\grids\lst_map\final_clean\480_based\city2023_lst_ratio_grid_120m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp

🔍 正在处理文件：D:\seoul\grids\lst_map\final_clean\480_based\city2023_lst_ratio_grid_240m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp
✔ 完成并保存：D:\seoul\grids\lst_map\final_clean\480_based\city2023_

# 00 画一下各个参数的图

In [24]:
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os
import numpy as np
from mapclassify import NaturalBreaks

# 输出文件夹
output_folder = r'D:\seoul\grids\lst_map\final_clean\480_based\figures\grids_colored'
os.makedirs(output_folder, exist_ok=True)
seoul_boundary_file = r'D:\seoul\c_data\Admin_boundary\Seoul_boundary.shp'

# 网格大小
grid_sizes = [120, 240, 480]

# 自动投影到韩国 UTM zone 52N（EPSG:32652）保证比例真实
target_crs = 'EPSG:32652'

for year in [2016, 2023]:
    explanatory_vars = [f'ext_{year}', f'nor_{year}', f'hr_{year}',
                        'BCR(%)', 'BHA', 'BHV', 'NDVI', 'SVF', 'EV(m)',
                        'Dist_BP', 'Dist_MT', 'Dist_WB', 'WR(%)']
    explanatory_vars_visual = [f'extreme', f'normal', f'HR',
                               'BCR', 'BHA', 'BHV', 'NDVI', 'SVF', 'EV',
                               'Dist_P', 'Dist_M', 'Dist_W', 'WR']

    for value_column, name in zip(explanatory_vars, explanatory_vars_visual):
        for i in grid_sizes:
            shp_path = rf'D:\seoul\grids\lst_map\final_clean\480_based\city{year}_lst_ratio_grid_{i}m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp'

            if not os.path.exists(shp_path):
                print(f"❌ 文件不存在：{shp_path}")
                continue

            gdf = gpd.read_file(shp_path)
            boundary = gpd.read_file(seoul_boundary_file)

            # 投影到目标 CRS（UTM zone 52N）
            if gdf.crs != target_crs:
                gdf = gdf.to_crs(target_crs)
            if boundary.crs != target_crs:
                boundary = boundary.to_crs(target_crs)

            # 创建分级分类器
            classifier = NaturalBreaks(gdf[value_column], k=5)
            bins = classifier.bins
            gdf['class'] = classifier.yb  # 分类索引

            # 色带
            cmap = plt.get_cmap('RdYlBu_r')
            colors = cmap(np.linspace(0, 1, 5))

            # 绘图
            fig, ax = plt.subplots(figsize=(9, 8))
            boundary.plot(ax=ax, color='white', edgecolor='black', linewidth=1.0, zorder=0)

            for cls in range(5):
                gdf[gdf['class'] == cls].plot(ax=ax, color=colors[cls], linewidth=0.2, edgecolor='gray', zorder=1)

            # ax.set_title(f'{year} {name} (Grid: {i}m)', fontsize=14)
            ax.axis('off')

            # 构建 legend
            patches = []
            for j in range(5):
                if j == 0:
                    label = f'≤ {bins[j]:.2f}'
                else:
                    label = f'{bins[j-1]:.2f} - {bins[j]:.2f}'
                patch = mpatches.Patch(color=colors[j], label=label)
                patches.append(patch)

            ax.legend(handles=patches,
                      title=name,
                      loc='upper left',
                      bbox_to_anchor=(0.85, 0.95),
                      frameon=True)

            # 保存图像
            output_path = os.path.join(output_folder, f'{year}_{name}_{i}m_map.png')
            fig.subplots_adjust(left=0.05, right=0.85, top=0.95, bottom=0.05)
            plt.savefig(output_path, dpi=300)
            plt.close()
            print(f"✅ Saved: {output_path}")


✅ Saved: D:\seoul\grids\lst_map\final_clean\480_based\figures\grids_colored\2016_extreme_120m_map.png
✅ Saved: D:\seoul\grids\lst_map\final_clean\480_based\figures\grids_colored\2016_extreme_240m_map.png
✅ Saved: D:\seoul\grids\lst_map\final_clean\480_based\figures\grids_colored\2016_extreme_480m_map.png
✅ Saved: D:\seoul\grids\lst_map\final_clean\480_based\figures\grids_colored\2016_normal_120m_map.png
✅ Saved: D:\seoul\grids\lst_map\final_clean\480_based\figures\grids_colored\2016_normal_240m_map.png
✅ Saved: D:\seoul\grids\lst_map\final_clean\480_based\figures\grids_colored\2016_normal_480m_map.png
✅ Saved: D:\seoul\grids\lst_map\final_clean\480_based\figures\grids_colored\2016_HR_120m_map.png
✅ Saved: D:\seoul\grids\lst_map\final_clean\480_based\figures\grids_colored\2016_HR_240m_map.png
✅ Saved: D:\seoul\grids\lst_map\final_clean\480_based\figures\grids_colored\2016_HR_480m_map.png
✅ Saved: D:\seoul\grids\lst_map\final_clean\480_based\figures\grids_colored\2016_BCR (%)_120m_map.pn

# 00 Descriptive statistics

In [165]:
import os
import re
import pandas as pd
import geopandas as gpd

grid_folder = r'D:\seoul\grids\lst_map\final_clean\480_based'
output_folder = os.path.join(grid_folder, r'statistics')
os.makedirs(output_folder, exist_ok=True)

for filename in os.listdir(grid_folder):
    if filename.endswith('_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp'):
        input_path = os.path.join(grid_folder, filename)
        data = gpd.read_file(input_path)

for year in [2016, 2023]:
    output_path = os.path.join(output_folder, f'{year} Descriptive statistics.xlsx')
    all_stats = []
    
    for filename in os.listdir(grid_folder):
        if filename.endswith('_xy.shp') and filename.startswith(f'city{year}'):
            input_path = os.path.join(grid_folder, filename)
            df = gpd.read_file(input_path)

            raw_columns = [f'nor_{year}', f'ext_{year}', f'hr_{year}', 'BHV', 'BCR', 'SVF', 'NDVI', 'EV', 'WR', 'Dist_P', 'Dist_W', 'Dist_M']
            output_labels = ['Normal_LST', 'Extreme_LST', 'HR_LST', 'BHV', 'BCR', 'SVF', 'NDVI', 'EV', 'WR', 'Dist_P', 'Dist_W', 'Dist_M']

            desc = df[raw_columns].describe().T[['mean', 'min', 'max', 'std']]
            desc.columns = ['Mean', 'Min', 'Max', 'SD']
            desc.index = output_labels
            desc = desc.round(2)

            # ✅ 提取 grid ID
            match = re.search(r'(\d{3,5})m', filename)
            grid_id = match.group(1) if match else 'Unknown'

            desc['Grid'] = grid_id

            # ✅ 添加观测值（每个 grid 有多少行）
            desc['N'] = len(df)

            all_stats.append(desc.reset_index())

    # 合并所有 grid 的结果
    summary_df = pd.concat(all_stats, ignore_index=True)
    summary_df.rename(columns={'index': 'Variable'}, inplace=True)

    # 输出 Excel
    summary_df.to_excel(output_path, index=False)
    print(f"📁 描述性统计合并完成！保存路径：{output_path}")


📁 描述性统计合并完成！保存路径：D:\seoul\grids\lst_map\final_clean\480_based\statistics\2016 Descriptive statistics.xlsx
📁 描述性统计合并完成！保存路径：D:\seoul\grids\lst_map\final_clean\480_based\statistics\2023 Descriptive statistics.xlsx


# 01 spatial regression model

## 00 先执行feature的correlation test, find out whether or not there has multicollinearity,用pearson, spearman还有传统VIF

① Pearson 相关系数（皮尔逊相关）
用于连续型变量之间的线性相关性检验。
范围：-1 到 1，越接近 ±1 说明线性关系越强。
适用：变量呈正态分布时。
from scipy.stats import pearsonr
r, p = pearsonr(x, y)

<!-- ② Spearman 等级相关系数（斯皮尔曼相关）-- 在我们的研究中适用
用于连续或顺序变量之间的单调关系检验，不要求正态分布。
适用：非正态分布或有异常值时更稳健。
from scipy.stats import spearmanr
rho, p = spearmanr(x, y) -->

③ VIF（方差膨胀因子，Variance Inflation Factor）
用于检测自变量之间的多重共线性。
规则：VIF > 10（或 5）表示严重共线性，需剔除相关变量。
from statsmodels.stats.outliers_influence import variance_inflation_factor
vif = variance_inflation_factor(X.values, i)

<!-- 比较出问题之后,剔除了BHA -->

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os
import geopandas as gpd
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor
from matplotlib.colors import ListedColormap

grid_folder = r'D:\seoul\grids\lst_map\final_clean\480_based'

# 自定义阈值颜色
colors = ["#ffffff", "#ffcc99", "#ff6666"]  # 低-中-高相关
cmap = ListedColormap(colors)

# 分类函数
def categorize_corr(r):
    r_abs = abs(r)
    if r_abs < 0.4:
        return 0
    elif r_abs < 0.8:
        return 1
    else:
        return 2

for year in [2016, 2023]:
    explanatory_vars = ['BCR', 'BHA', 'BHV', 'SVF', 'NDVI', 'EV', 'WR', 'Dist_W', 'Dist_P', 'Dist_M']
    for filename in os.listdir(grid_folder):
        if filename.endswith('480m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp') and filename.startswith(f'city{year}'):
            path = os.path.join(grid_folder, filename)
            gdf = gpd.read_file(path)
            df = gdf[explanatory_vars].dropna()

            # ----------------- 相关性 -----------------
            pearson_corr = df.corr(method='pearson')
            spearman_corr = df.corr(method='spearman')

            # 分类矩阵
            pearson_cat = pearson_corr.applymap(categorize_corr)
            spearman_cat = spearman_corr.applymap(categorize_corr)

            mask = np.triu(np.ones_like(pearson_corr, dtype=bool))  # 上三角遮掉

            # --- Pearson ---
            fig, ax = plt.subplots(figsize=(10, 8))
            sns.heatmap(
                pearson_cat,
                annot=pearson_corr.round(2),
                fmt=".2f",
                cmap=cmap,
                ax=ax,
                mask=mask,
                cbar_kws={"ticks": [0.33, 1, 1.66], "label": "Correlation Level"}
            )
            plt.title(f"Pearson Correlation Matrix ({year})\nThresholds: 0.4 (moderate), 0.8 (high)")
            plt.tight_layout()
            save_path = os.path.join(grid_folder, r'statistics\fig\variable_correlation', f"{year}_pearson_corr_cat.png")
            plt.savefig(save_path)
            plt.close()

            # --- Spearman ---
            fig, ax = plt.subplots(figsize=(10, 8))
            sns.heatmap(
                spearman_cat,
                annot=spearman_corr.round(2),
                fmt=".2f",
                cmap=cmap,
                ax=ax,
                mask=mask,
                cbar_kws={"ticks": [0.33, 1, 1.66], "label": "Correlation Level"}
            )
            plt.title(f"Spearman Correlation Matrix ({year})\nThresholds: 0.4 (moderate), 0.8 (high)")
            plt.tight_layout()
            save_path = os.path.join(grid_folder, r'statistics\fig\variable_correlation', f"{year}_spearman_corr_cat.png")
            plt.savefig(save_path)
            plt.close()

            # ----------------- VIF -----------------
            df_vif = df.copy()
            df_vif["Intercept"] = 1
            vif_df = pd.DataFrame()
            vif_df["Variable"] = df_vif.columns
            vif_df["VIF"] = [variance_inflation_factor(df_vif.values, i) for i in range(df_vif.shape[1])]
            vif_df = vif_df[vif_df["Variable"] != "Intercept"]

            vif_path = os.path.join(grid_folder, r'statistics\fig\variable_correlation', f"{year}_VIF.xlsx")
            vif_df.to_excel(vif_path, index=False)

            print(f"✅ 变量相关性分析完成：{year}")


C:\Users\owner\AppData\Local\Temp\ipykernel_30688\1462696222.py:39: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  pearson_cat = pearson_corr.applymap(categorize_corr)
C:\Users\owner\AppData\Local\Temp\ipykernel_30688\1462696222.py:40: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  spearman_cat = spearman_corr.applymap(categorize_corr)


✅ 变量相关性分析完成：2016


C:\Users\owner\AppData\Local\Temp\ipykernel_30688\1462696222.py:39: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  pearson_cat = pearson_corr.applymap(categorize_corr)
C:\Users\owner\AppData\Local\Temp\ipykernel_30688\1462696222.py:40: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  spearman_cat = spearman_corr.applymap(categorize_corr)


✅ 变量相关性分析完成：2023


## 01 OLS regression models.

首先使用普通最小二乘法（Ordinary Least Squares, OLS）建立基础回归模型。
评估模型解释力和残差行为。

In [174]:
import os
import geopandas as gpd
import statsmodels.api as sm
import re
import numpy as np
from spreg import OLS, LMtests

# 📁 文件夹路径
grid_folder = r'D:\seoul\grids\lst_map\final_clean\480_based'


for year in [2016,2023]:

    target_vars = [f'nor_{year}', f'ext_{year}', f'hr_{year}']
    explanatory_vars = ['BCR', 'BHV',  'SVF', 'NDVI', 'EV', 'WR', 'Dist_W', 'Dist_P', 'Dist_M']    # 顺序很讲究
    for filename in os.listdir(grid_folder):
        if filename.endswith('480m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp') and filename.startswith(f'city{year}'):
            filepath = os.path.join(grid_folder, filename)
            print(f"Processing {filename}...")

            # 读取 shapefile
            gdf = gpd.read_file(filepath)

            for target in target_vars:
                y = gdf[target]
                X = gdf[explanatory_vars]
                '''
                y = β₁X₁ + β₂X₂ + ...
                '''
                X = sm.add_constant(X)  # 添加截距项
                ''' -> 截距的意思是
                y = β₁X₁ + β₂X₂ + ...+ ε
                '''

                model = sm.OLS(y, X)
                results = model.fit()

                # 提取文件名中的 3~5 位数字
                match = re.search(r'grid_(\d{3,5})m', filename)
                file_id = match.group() if match else "Unknown"

                print(f"OLS results for {file_id} | target: {target}")
                #print(results.summary())

                print(f"R²: {results.rsquared:.4f}")
                rmse = np.sqrt(np.mean(results.resid ** 2))
                print(f"RMSE: {rmse:.4f}")

Processing city2016_lst_ratio_grid_480m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp...
OLS results for grid_480m | target: nor_2016
R²: 0.8539
RMSE: 0.9577
OLS results for grid_480m | target: ext_2016
R²: 0.9055
RMSE: 1.2423
OLS results for grid_480m | target: hr_2016
R²: 0.8102
RMSE: 0.7770
Processing city2023_lst_ratio_grid_480m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp...
OLS results for grid_480m | target: nor_2023
R²: 0.9206
RMSE: 1.1580
OLS results for grid_480m | target: ext_2023
R²: 0.8622
RMSE: 1.6241
OLS results for grid_480m | target: hr_2023
R²: 0.1298
RMSE: 1.3000


In [ ]:
#数据自身的统计情况
# import geopandas as gpd
# import rasterio
# import numpy as np
# from rasterio.mask import mask
# import pandas as pd
# import os

# # ------------------- 公共函数 -------------------
# def raster_stats(raster_path, shapes):
#     with rasterio.open(raster_path) as src:
#         shapes = shapes.to_crs(src.crs)
#         try:
#             out_image, _ = mask(src, shapes.geometry, crop=True)
#         except ValueError:
#             return np.nan, np.nan, np.nan, np.nan  # 完全不重叠

#         data = out_image[0]
#         data = data[data != src.nodata]
#         if len(data) == 0:
#             return np.nan, np.nan, np.nan, np.nan
#         return data.mean(), data.min(), data.max(), data.std()

# def single_val_stats(val):
#     return val, np.nan, np.nan, np.nan

# def coverage_ratio(feature_gdf, valid_gdf, grid_area):
#     valid_gdf = valid_gdf.to_crs(feature_gdf.crs)
#     inside_feature = gpd.overlay(feature_gdf, valid_gdf, how='intersection')
#     return (inside_feature.geometry.area.sum() / grid_area) * 100

# # ------------------- 主函数 -------------------
# def calculate_summary(year,
#                       shp_valid_path,
#                       building_path,
#                       ndvi_path,
#                       svf_path,
#                       elevation_path,
#                       normal_path,
#                       extreme_path,
#                       hr_path,
#                       water_path,
#                       output_folder):

#     # --- shapefile ---
#     valid_gdf = gpd.read_file(shp_valid_path)
#     building_gdf = gpd.read_file(building_path)
#     water_gdf = gpd.read_file(water_path)

#     # --- 建筑物 ---
#     valid_gdf = valid_gdf.to_crs(building_gdf.crs)
#     inside_building = gpd.overlay(building_gdf, valid_gdf, how='intersection')

#     if 'area' not in inside_building.columns:
#         inside_building['area'] = inside_building.geometry.area

#     total_bldg_area = inside_building['area'].sum()
#     height_std = inside_building['height'].std()
#     height_std = 0 if np.isnan(height_std) else height_std
#     grid_area = valid_gdf.geometry.area.sum()
#     bcr = (total_bldg_area / grid_area) * 100
#     water_ratio = coverage_ratio(water_gdf, valid_gdf, grid_area)

#     # --- 栅格 ---
#     ndvi_stats = raster_stats(ndvi_path, valid_gdf)
#     svf_stats = raster_stats(svf_path, valid_gdf)
#     dem_stats = raster_stats(elevation_path, valid_gdf)
#     normalheat_stats = raster_stats(normal_path, valid_gdf)
#     extremeheat_stats = raster_stats(extreme_path, valid_gdf)
#     hr_stats = raster_stats(hr_path, valid_gdf)

#     # --- 单值 ---
#     bcr_stats = single_val_stats(bcr)
#     bhv_stats = single_val_stats(height_std)
#     wr_stats = single_val_stats(water_ratio)

#     # --- 构建 summary ---
#     summary_values = [
#         *normalheat_stats,
#         *extremeheat_stats,
#         *hr_stats,
#         *bhv_stats,
#         *bcr_stats,
#         *svf_stats,
#         *ndvi_stats,
#         *dem_stats,
#         *wr_stats
#     ]

#     metrics = [
#         "Normal_LST", "Extreme_LST", "HR_T", "BHV", "BCR(%)", 
#         "SVF", "NDVI", "EV", "WR(%)"
#     ]

#     rows = []
#     for i, metric in enumerate(metrics):
#         start = i * 4
#         stats_group = summary_values[start:start+4]
#         rows.append([metric, *stats_group])

#     summary_matrix = pd.DataFrame(
#         rows,
#         columns=["Indicator", "Mean", "Min", "Max", "Std"]
#     )

#     # --- 保存 ---
#     os.makedirs(output_folder, exist_ok=True)
#     output_file = os.path.join(output_folder, f"seoul_{year}_summary.xlsx")
#     summary_matrix.to_excel(output_file, index=False)

#     print(f"\n====== {year} Summary Statistics (Grouped) ======")
#     print(summary_matrix.to_string(index=False))
#     print(f"\n✅ 已保存到 {output_file}")
#     return summary_matrix


# # 
# calculate_summary(
#     2016,
#     shp_valid_path=r"D:\seoul\grids\lst_map\final_clean\480_based\valid_2016_grid_480m.shp",
#     building_path=r"D:\seoul\c_data\a_building\final\Fin_building.shp",
#     ndvi_path=r"D:\seoul\c_data\b_ndvi\d_clean\clipped_ndvi_20160924_cleaned.tif",
#     svf_path=r"D:\seoul\c_data\c_svf\final_svf.tif",
#     elevation_path=r"D:\seoul\c_data\d_elevation\seoul_dem.tif",
#     normal_path=r"D:\seoul\b_satellite_img\normal_heat_day_lst\clipped_20160924_B10_cleaned_LST.tif",
#     extreme_path=r"D:\seoul\b_satellite_img\extreme_heat_day_lst\clipped_20160807_B10_cleaned_LST.tif",
#     hr_path=r"D:\seoul\b_satellite_img\heat_resilience\2016_heat_resilience.tif",
#     water_path=r"D:\seoul\c_data\g_stream\seoul_stream_final.shp",
#     output_folder=r"D:\seoul\grids\lst_map\final_clean\480_based\statistics\original data statistics"
# )

# calculate_summary(
#     2023,
#     shp_valid_path=r"D:\seoul\grids\lst_map\final_clean\480_based\valid_2023_grid_480m.shp",
#     building_path=r"D:\seoul\c_data\a_building\final\Fin_building.shp",
#     ndvi_path=r"D:\seoul\c_data\b_ndvi\d_clean\clipped_ndvi_20230616_cleaned.tif",
#     svf_path=r"D:\seoul\c_data\c_svf\final_svf.tif",
#     elevation_path=r"D:\seoul\c_data\d_elevation\seoul_dem.tif",
#     normal_path=r"D:\seoul\b_satellite_img\normal_heat_day_lst\clipped_20230616_B10_cleaned_LST.tif",
#     extreme_path=r"D:\seoul\b_satellite_img\extreme_heat_day_lst\clipped_20230819_B10_cleaned_LST.tif",
#     hr_path=r"D:\seoul\b_satellite_img\heat_resilience\2023_heat_resilience.tif",
#     water_path=r"D:\seoul\c_data\g_stream\seoul_stream_final.shp",
#     output_folder=r"D:\seoul\grids\lst_map\final_clean\480_based\statistics\original data statistics"
# )


====== 2016 Summary Statistics (Grouped) ======
  Indicator       Mean        Min        Max       Std
 Normal_LST  31.050320  22.452295  45.667488  2.883915
Extreme_LST  41.600376  28.806395  58.717487  4.570520
       HR_T -10.556202 -16.683355  -2.187532  2.027864
        BHV   9.927906        NaN        NaN       NaN
     BCR(%)  18.791399        NaN        NaN       NaN
        SVF   0.857052   0.010926   1.000000  0.154404
       NDVI   0.151221  -0.082758   0.495363  0.096151
         EV  48.417404  -5.000000 628.864685 60.330868
      WR(%)   6.151363        NaN        NaN       NaN

✅ 已保存到 D:\seoul\grids\lst_map\final_clean\480_based\statistics\original data statistics\seoul_2016_summary.xlsx

====== 2023 Summary Statistics (Grouped) ======
  Indicator      Mean        Min        Max       Std
 Normal_LST 37.249454  24.697935  53.183712  4.674873
Extreme_LST 41.981354  29.626720  59.411346  4.947564
       HR_T -4.734814 -12.079281   5.943935  1.516230
        BHV  9.832980  

,Indicator,Mean,Min,Max,Std
0,Normal_LST,37.249454,24.697935,53.183712,4.674873
1,Extreme_LST,41.981354,29.626720,59.411346,4.947564
2,HR_T,-4.734814,-12.079281,5.943935,1.516230
3,BHV,9.832980,NaN,NaN,NaN
4,BCR(%),18.029491,NaN,NaN,NaN
5,SVF,0.864290,0.010926,1.000000,0.151518
6,NDVI,0.196386,-0.078430,0.539992,0.131254
7,EV,46.477734,-5.000000,535.000000,54.552383
8,WR(%),6.289498,NaN,NaN,NaN


## 02 Moran’s I statistics:Test for Spatial Autocorrelation

使用 Moran’s I 统计量对 OLS 残差进行空间自相关检验。
若结果显著，说明存在空间自相关，OLS 模型不适用，需考虑空间回归模型。

In [29]:
import sys
# ! {sys.executable} -m pip install libpysal==4.6.0
import libpysal
print(libpysal.__version__)
# ! python.exe -m pip install --upgrade pip


4.13.0


In [ ]:
import os
import geopandas as gpd
import numpy as np
from esda.moran import Moran
from libpysal.weights import Queen
import matplotlib.pyplot as plt
from scipy.sparse.csgraph import connected_components

grid_folder = r'D:\seoul\grids\lst_map\final_clean\480_based'

for year in [2016, 2023]:
    target_vars = [f'nor_{year}', f'ext_{year}', f'hr_{year}']
    explanatory_vars = ['BCR', 'BHV',  'SVF', 'NDVI', 'EV', 'WR', 'Dist_W', 'Dist_P', 'Dist_M']    # 顺序很讲究
    for filename in os.listdir(grid_folder):
        if filename.endswith('480m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp') and filename.startswith(f'city{year}'):
            filepath = os.path.join(grid_folder, filename)
            print(f"\n📂 Processing {filename}...")

            gdf = gpd.read_file(filepath)
        
            for target in target_vars:
                print(f"\n🔍 Analyzing variable: {target}")

                # 构建空间权重并移除孤岛
                w = Queen.from_dataframe(gdf, use_index=False)
                islands = [i for i, neighbors in w.neighbors.items() if len(neighbors) == 0]

                gdf_clean = gdf.drop(index=islands).reset_index(drop=True)
                # 构建权重矩阵
                w_clean = Queen.from_dataframe(gdf_clean, use_index=False)

                # 转为邻接矩阵
                adj_matrix = w_clean.full()[0]

                # 识别连通分量
                n_components, labels = connected_components(csgraph=adj_matrix, directed=False)

                # 找出最大连通分量的标签
                largest_label = np.bincount(labels).argmax()

                # 找出对应索引
                main_component = np.where(labels == largest_label)[0]

                # 筛选最大连通子图
                gdf_main = gdf_clean.iloc[main_component].reset_index(drop=True)

                # 重建空间权重矩阵
                w_main = Queen.from_dataframe(gdf_main, use_index=False)
                w_main.transform = 'r'

                y = gdf_main[target]

                # 计算 Moran's I
                moran = Moran(y, w_main)
        
                print(f"Moran's I: {moran.I:.4f}")
                print(f"p-value: {moran.p_sim:.4f}")
                print(f"Expected I: {moran.EI:.4f}")
                print(f"Variance: {moran.VI_sim:.4f}")

                if moran.p_sim < 0.05:
                    print(f"'{year}✅ Significant spatial autocorrelation detected.")
                else:
                    print("❌ No significant spatial autocorrelation detected.")
                


📂 Processing city2016_lst_ratio_grid_480m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp...

🔍 Analyzing variable: nor_2016


c:\Users\owner\Python\Python311\Lib\site-packages\libpysal\weights\contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 12 disconnected components.
 There are 6 islands with ids: 1308, 1377, 1411, 1471, 1525, 1690.
  W.__init__(self, neighbors, ids=ids, **kw)
c:\Users\owner\Python\Python311\Lib\site-packages\libpysal\weights\contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 6 disconnected components.
  W.__init__(self, neighbors, ids=ids, **kw)


Moran's I: 0.6881
p-value: 0.0010
Expected I: -0.0006
Variance: 0.0002
'2016✅ Significant spatial autocorrelation detected.

🔍 Analyzing variable: ext_2016


c:\Users\owner\Python\Python311\Lib\site-packages\libpysal\weights\contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 12 disconnected components.
 There are 6 islands with ids: 1308, 1377, 1411, 1471, 1525, 1690.
  W.__init__(self, neighbors, ids=ids, **kw)
c:\Users\owner\Python\Python311\Lib\site-packages\libpysal\weights\contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 6 disconnected components.
  W.__init__(self, neighbors, ids=ids, **kw)


Moran's I: 0.6760
p-value: 0.0010
Expected I: -0.0006
Variance: 0.0002
'2016✅ Significant spatial autocorrelation detected.

🔍 Analyzing variable: hr_2016


...

Moran's I: 0.6775
p-value: 0.0010
Expected I: -0.0007
Variance: 0.0002
'2023✅ Significant spatial autocorrelation detected.

🔍 Analyzing variable: ext_2023


c:\Users\owner\Python\Python311\Lib\site-packages\libpysal\weights\contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 8 disconnected components.
 There are 2 islands with ids: 1392, 1571.
  W.__init__(self, neighbors, ids=ids, **kw)
c:\Users\owner\Python\Python311\Lib\site-packages\libpysal\weights\contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 6 disconnected components.
  W.__init__(self, neighbors, ids=ids, **kw)


Moran's I: 0.7007
p-value: 0.0010
Expected I: -0.0007
Variance: 0.0002
'2023✅ Significant spatial autocorrelation detected.

🔍 Analyzing variable: hr_2023


c:\Users\owner\Python\Python311\Lib\site-packages\libpysal\weights\contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 8 disconnected components.
 There are 2 islands with ids: 1392, 1571.
  W.__init__(self, neighbors, ids=ids, **kw)
c:\Users\owner\Python\Python311\Lib\site-packages\libpysal\weights\contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 6 disconnected components.
  W.__init__(self, neighbors, ids=ids, **kw)


Moran's I: 0.8606
p-value: 0.0010
Expected I: -0.0007
Variance: 0.0002
'2023✅ Significant spatial autocorrelation detected.


## 03 LM Test: Identify the Type of Spatial Dependence
使用 Lagrange Multiplier（LM）检验 判断空间依赖类型：
* LM-Lag（滞后型）：是否存在因变量的空间滞后项。
* LM-Error（误差型）：是否存在误差项的空间相关性。
* 同时参考 Robust LM-Lag 和 Robust LM-Error（控制另一种依赖类型后的检验）。
* 如果两个 Robust LM 检验都显著，说明两种依赖都可能存在，建议使用更一般形式的模型，如 SDM。

In [ ]:
import os
import numpy as np
import geopandas as gpd
import libpysal
from spreg import OLS, LMtests

grid_folder = r'D:\seoul\grids\lst_map\final_clean\480_based'

for year in [2016, 2023]:
    target_vars = [f'nor_{year}', f'ext_{year}', f'hr_{year}']
    explanatory_vars = ['BCR(%)', 'BHV', 'NDVI', 'SVF', 'EV(m)',
                        'Dist_BP', 'Dist_MT', 'Dist_WB', 'WR(%)']
    # explanatory_vars = ['BCR(%)', 'BHV', 'NDVI', 'SVF', 'EV(m)',
    #                     'WR(%)']
    for filename in os.listdir(grid_folder):
        if filename.endswith('480m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp') and filename.startswith(f'city{year}'):
            filepath = os.path.join(grid_folder, filename)
            print(f"\nProcessing file: {filename}")

            gdf = gpd.read_file(filepath)
            w = libpysal.weights.Queen.from_dataframe(gdf, use_index=False)
            w.transform = 'r'

            for target in target_vars:
                print(f"\n→ Target variable: {target}")

                y = gdf[target].values.reshape(-1, 1)  # 变成二维数组
                X = gdf[explanatory_vars].values
                X = np.hstack((np.ones((X.shape[0], 1)), X))  # 添加常数项

                model = OLS(y, X, w=w, spat_diag=True, name_y=target, name_x=['const'] + explanatory_vars)
                
                # LMtests需要的是spreg的OLS结果，直接从model拿
                lms = LMtests(model, w)
                print(dir(lms))

                print(f"  LM-Lag (lml)                   : {lms.lml[0]:.4f}, p-value = {lms.lml[1]:.4g}")
                print(f"  LM-Error (lme)                 : {lms.lme[0]:.4f}, p-value = {lms.lme[1]:.4g}")
                print(f"  Robust LM-Lag (rlml)           : {lms.rlml[0]:.4f}, p-value = {lms.rlml[1]:.4g}")
                print(f"  Robust LM-Error (rlme)         : {lms.rlme[0]:.4f}, p-value = {lms.rlme[1]:.4g}")


Processing file: city2016_lst_ratio_grid_480m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp


c:\Users\owner\Python\Python311\Lib\site-packages\libpysal\weights\contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 12 disconnected components.
 There are 6 islands with ids: 1308, 1377, 1411, 1471, 1525, 1690.
  W.__init__(self, neighbors, ids=ids, **kw)


('WARNING: ', 1308, ' is an island (no neighbors)')
('WARNING: ', 1377, ' is an island (no neighbors)')
('WARNING: ', 1411, ' is an island (no neighbors)')
('WARNING: ', 1471, ' is an island (no neighbors)')
('WARNING: ', 1525, ' is an island (no neighbors)')
('WARNING: ', 1690, ' is an island (no neighbors)')

→ Target variable: nor_2016
['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', 'lme', 'lml', 'lmslxerr', 'lmspdurbin', 'lmwx', 'rlmdurlag', 'rlme', 'rlml', 'rlmwx', 'sarma']
  LM-Lag (lml)                   : 211.3825, p-value = 6.859e-48
  LM-Error (lme)                 : 1652.8657, p-value = 0
  Robust LM-Lag (rlml)           : 16.2983, p-value = 5.411e-05
  Robust LM-Erro

c:\Users\owner\Python\Python311\Lib\site-packages\libpysal\weights\contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 8 disconnected components.
 There are 2 islands with ids: 1392, 1571.
  W.__init__(self, neighbors, ids=ids, **kw)


## 04 Model Comparison: Likelihood Ratio (LR) Tests
在模型之间进行似然比检验（Likelihood Ratio Test），判断引入空间项后模型改进是否显著。

In [34]:
# import sys
# !{sys.executable} -m pip install --upgrade spreg
# !{sys.executable} -m pip install --upgrade libpysal
# import spreg
# print(spreg.__version__)

In [ ]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import libpysal
from spreg import OLS, ML_Lag, ML_Error
from spreg.diagnostics import likratiotest
import time
from scipy.stats import chi2

# 文件夹路径
grid_folder = r'D:\seoul\grids\lst_map\final_clean\480_based'

# 初始化 DataFrame 保存结果
lr_results = pd.DataFrame(columns=[
    'Year', 'City', 'Dependent', 'Test', 'LR_stat', 'df', 'p_value'
])
model_info = pd.DataFrame(columns=[
    'Year', 'City', 'Dependent', 'Model', 'Variable', 'Coef', 'Std_err', 't_stat', 'p_value', 'R2', 'Adj_R2', 'Rho'
])

# 循环年份
for year in [2016, 2023]:
    target_vars = [f'nor_{year}', f'ext_{year}', f'hr_{year}']
    explanatory_vars = ['BCR(%)', 'BHV', 'NDVI', 'SVF', 'EV(m)',
                        'Dist_BP', 'Dist_MT', 'Dist_WB', 'WR(%)']

    for filename in os.listdir(grid_folder):
        if filename.endswith('480m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp') and filename.startswith(f'city{year}'):
            path = os.path.join(grid_folder, filename)
            print(f"\nProcessing {filename}")
            gdf = gpd.read_file(path)

            # 构造空间权重矩阵（1000米阈值距离）
            threshold = 1000
            w = libpysal.weights.DistanceBand.from_dataframe(gdf, threshold=threshold, binary=False)
            w.transform = 'r'

            # 构造 y, X
            X0 = gdf[explanatory_vars].values
            x = np.hstack([np.ones((len(gdf), 1)), X0])  # 加常数列

            city_name = filename.split('_')[0]
            w_name = 'W1000'
            ds_name = f'yr{year}'

            for target in target_vars:
                yi = gdf[[target]].values
                models = {}

                # 1. OLS
                models['OLS'] = OLS(yi, x, w=w, spat_diag=True, moran=True,
                                    name_w=w_name, name_ds=ds_name)
                # SLX
                models['SLX'] = OLS(yi, x, w=w, slx_lags=1, spat_diag=True, moran=True,
                 name_w=w_name,name_ds=ds_name)
                
                # 2. SLM
                models['SLM'] = ML_Lag(yi, x, w=w, method="full",
                                       name_w=w_name, name_ds=ds_name,
                                       spat_impacts=['full'])

                # 3. SDM
                models['SDM'] = ML_Lag(yi, x, w=w, slx_lags=1,
                                       name_w=w_name, name_ds=ds_name,
                                       spat_impacts=['full'],
                                       spat_diag=True)

                # 4. SEM
                models['SEM']  = ML_Error(yi, x, w=w, method="full",
                     name_w=w_name,name_ds=ds_name)

                # 5. SDEM
                models['SDEM']  = ML_Error(yi, x, w=w, slx_lags=1, method="full",
                    name_w=w_name, name_ds=ds_name)


                # 保存模型系数信息
                for mdl_name, mdl in models.items():
                    coef = mdl.betas.flatten()
                    var_names = ['const'] + explanatory_vars
                    if mdl_name in ['SLM', 'SDM']:
                        var_names += [f"W_{v}" for v in explanatory_vars]
                    if mdl_name == 'SDEM':
                        var_names += [f"W_{v}" for v in explanatory_vars]  # SDEM Durbin
                    se = mdl.std_err.flatten()
                    t_stat = coef / se
                    p_val = 2 * (1 - chi2.cdf(t_stat**2, 1))
                    R2 = getattr(mdl, 'r2', np.nan)
                    R2_adj = getattr(mdl, 'ar2', np.nan)
                    Rho = getattr(mdl, 'rho', np.nan)
                    if mdl_name in ['SEM', 'SDEM']:
                        Rho = getattr(mdl, 'lambda', np.nan)

                    for c, v, s, t, p in zip(coef, var_names, se, t_stat, p_val):
                        model_info = pd.concat([model_info, pd.DataFrame([{
                            'Year': year,
                            'City': city_name,
                            'Dependent': target,
                            'Model': mdl_name,
                            'Variable': v,
                            'Coef': c,
                            'Std_err': s,
                            't_stat': t,
                            'p_value': p,
                            'R2': R2,
                            'Adj_R2': R2_adj,
                            'Rho': Rho
                        }])], ignore_index=True)
                def safe_likratiotest(m0, m1):
                    res = likratiotest(m0, m1)
                    df_expected = len(m1.betas.flatten()) - len(m0.betas.flatten())
                    if res['df'] != df_expected:
                        print(f"⚠️ df mismatch: likratiotest={res['df']}, expected={df_expected}")
                        res['df'] = df_expected
                    return res
                # 提取 LR 检验
                lr_tests = {
                    'SLX-OLS': safe_likratiotest(models['OLS'], models['SLX']),
                    'SLM-OLS': safe_likratiotest(models['OLS'], models['SLM']),
                    'SDM-OLS': safe_likratiotest(models['OLS'], models['SDM']),
                    'SEM-OLS': safe_likratiotest(models['OLS'], models['SEM']),
                    'SDM-SLX': safe_likratiotest(models['SLX'], models['SDM']),
                    'SDM-SLM': safe_likratiotest(models['SLM'], models['SDM']),
                    'SDM-SEM': safe_likratiotest(models['SEM'], models['SDM']),
                    'SDEM-SLX': safe_likratiotest(models['SLX'], models['SDEM']),
                    'SDEM-SEM': safe_likratiotest(models['SEM'], models['SDEM']),
                }

                for test_name, res in lr_tests.items():
                    lr_results = pd.concat([lr_results, pd.DataFrame([{
                        'Year': year,
                        'City': city_name,
                        'Dependent': target,
                        'Test': test_name,
                        'LR_stat': res['likr'],
                        'df': res['df'],
                        'p_value': res['p-value']
                    }])], ignore_index=True)
                    print(f"{test_name}: df(likratiotest)={res['df']}, "
                            f"params_m0={len(models[test_name.split('-')[0]].betas.flatten())}, "
                            f"params_m1={len(models[test_name.split('-')[1]].betas.flatten())}")



# 保存 Excel
output_path = r'D:\seoul\grids\lst_map\final_clean\480_based\statistics\LR_test_results.xlsx'
with pd.ExcelWriter(output_path) as writer:
    lr_results.to_excel(writer, sheet_name='LR_tests', index=False)
    model_info.to_excel(writer, sheet_name='Model_info', index=False)


print(f"All LR test results and model info saved to {output_path}")


Processing city2016_lst_ratio_grid_480m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp


c:\Users\owner\Python\Python311\Lib\site-packages\scipy\sparse\_data.py:133: RuntimeWarning: divide by zero encountered in reciprocal
  return self._with_data(data ** n)
c:\Users\owner\Python\Python311\Lib\site-packages\libpysal\weights\util.py:826: UserWarning: The weights matrix is not fully connected: 
 There are 12 disconnected components.
 There are 6 islands with ids: 1308, 1377, 1411, 1471, 1525, 1690.
  w = W(neighbors, weights, ids, **kwargs)
c:\Users\owner\Python\Python311\Lib\site-packages\libpysal\weights\distance.py:844: UserWarning: The weights matrix is not fully connected: 
 There are 12 disconnected components.
 There are 6 islands with ids: 1308, 1377, 1411, 1471, 1525, 1690.
  W.__init__(


('WARNING: ', 1308, ' is an island (no neighbors)')
('WARNING: ', 1377, ' is an island (no neighbors)')
('WARNING: ', 1411, ' is an island (no neighbors)')
('WARNING: ', 1471, ' is an island (no neighbors)')
('WARNING: ', 1525, ' is an island (no neighbors)')
('WARNING: ', 1690, ' is an island (no neighbors)')


c:\Users\owner\Python\Python311\Lib\site-packages\spreg\ml_error.py:184: RuntimeWarning: Method 'bounded' does not support relative tolerance in x; defaulting to absolute tolerance.
  res = minimize_scalar(
c:\Users\owner\Python\Python311\Lib\site-packages\spreg\ml_error.py:184: RuntimeWarning: Method 'bounded' does not support relative tolerance in x; defaulting to absolute tolerance.
  res = minimize_scalar(
C:\Users\owner\AppData\Local\Temp\ipykernel_29800\2994932656.py:96: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  model_info = pd.concat([model_info, pd.DataFrame([{
C:\Users\owner\AppData\Local\Temp\ipykernel_29800\2994932656.py:131: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In

⚠️ df mismatch: likratiotest=10, expected=9
SLX-OLS: df(likratiotest)=9, params_m0=19, params_m1=10
SLM-OLS: df(likratiotest)=1, params_m0=11, params_m1=10
SDM-OLS: df(likratiotest)=10, params_m0=20, params_m1=10
SEM-OLS: df(likratiotest)=1, params_m0=11, params_m1=10
SDM-SLX: df(likratiotest)=1, params_m0=20, params_m1=19
SDM-SLM: df(likratiotest)=9, params_m0=20, params_m1=11
SDM-SEM: df(likratiotest)=9, params_m0=20, params_m1=11
SDEM-SLX: df(likratiotest)=1, params_m0=20, params_m1=19
SDEM-SEM: df(likratiotest)=9, params_m0=20, params_m1=11


...

⚠️ df mismatch: likratiotest=10, expected=9
SLX-OLS: df(likratiotest)=9, params_m0=19, params_m1=10
SLM-OLS: df(likratiotest)=1, params_m0=11, params_m1=10
SDM-OLS: df(likratiotest)=10, params_m0=20, params_m1=10
SEM-OLS: df(likratiotest)=1, params_m0=11, params_m1=10
SDM-SLX: df(likratiotest)=1, params_m0=20, params_m1=19
SDM-SLM: df(likratiotest)=9, params_m0=20, params_m1=11
SDM-SEM: df(likratiotest)=9, params_m0=20, params_m1=11
SDEM-SLX: df(likratiotest)=1, params_m0=20, params_m1=19
SDEM-SEM: df(likratiotest)=9, params_m0=20, params_m1=11


c:\Users\owner\Python\Python311\Lib\site-packages\spreg\ml_error.py:184: RuntimeWarning: Method 'bounded' does not support relative tolerance in x; defaulting to absolute tolerance.
  res = minimize_scalar(
c:\Users\owner\Python\Python311\Lib\site-packages\spreg\ml_error.py:184: RuntimeWarning: Method 'bounded' does not support relative tolerance in x; defaulting to absolute tolerance.
  res = minimize_scalar(


⚠️ df mismatch: likratiotest=10, expected=9
SLX-OLS: df(likratiotest)=9, params_m0=19, params_m1=10
SLM-OLS: df(likratiotest)=1, params_m0=11, params_m1=10
SDM-OLS: df(likratiotest)=10, params_m0=20, params_m1=10
SEM-OLS: df(likratiotest)=1, params_m0=11, params_m1=10
SDM-SLX: df(likratiotest)=1, params_m0=20, params_m1=19
SDM-SLM: df(likratiotest)=9, params_m0=20, params_m1=11
SDM-SEM: df(likratiotest)=9, params_m0=20, params_m1=11
SDEM-SLX: df(likratiotest)=1, params_m0=20, params_m1=19
SDEM-SEM: df(likratiotest)=9, params_m0=20, params_m1=11


c:\Users\owner\Python\Python311\Lib\site-packages\spreg\ml_error.py:184: RuntimeWarning: Method 'bounded' does not support relative tolerance in x; defaulting to absolute tolerance.
  res = minimize_scalar(
c:\Users\owner\Python\Python311\Lib\site-packages\spreg\ml_error.py:184: RuntimeWarning: Method 'bounded' does not support relative tolerance in x; defaulting to absolute tolerance.
  res = minimize_scalar(


⚠️ df mismatch: likratiotest=10, expected=9
SLX-OLS: df(likratiotest)=9, params_m0=19, params_m1=10
SLM-OLS: df(likratiotest)=1, params_m0=11, params_m1=10
SDM-OLS: df(likratiotest)=10, params_m0=20, params_m1=10
SEM-OLS: df(likratiotest)=1, params_m0=11, params_m1=10
SDM-SLX: df(likratiotest)=1, params_m0=20, params_m1=19
SDM-SLM: df(likratiotest)=9, params_m0=20, params_m1=11
SDM-SEM: df(likratiotest)=9, params_m0=20, params_m1=11
SDEM-SLX: df(likratiotest)=1, params_m0=20, params_m1=19
SDEM-SEM: df(likratiotest)=9, params_m0=20, params_m1=11
All LR test results and model info saved to D:\seoul\grids\lst_map\final_clean\480_based\statistics\LR_test_results.xlsx


## 00+ (AIC BIC 间接分析)需要一个方法比较SEM和SDM

In [133]:
# import numpy as np
# import pandas as pd
# import geopandas as gpd
# from shapely.geometry import Point
# import libpysal

# # ===== 1) 构造一个小测试数据（5个点，单位：米）=====
# # 如果你有真实 data_gdf（且是投影到米），直接用：
# # data_gdf = your_gdf.to_crs(XXXX)
# coords = [
#     (0, 0),
#     (300, 0),
#     (800, 0),
#     (0, 900),
#     (1200, 0),
# ]
# gdf = gpd.GeoDataFrame(geometry=[Point(xy) for xy in coords], crs="EPSG:5186")  # 例子用米制CRS

# threshold = 1000  # 1000米邻居阈值

# # ===== 2) 生成两种权重：binary 等权 vs 非 binary 衰减 =====
# w_bin = libpysal.weights.DistanceBand.from_dataframe(gdf, threshold=threshold, binary=True)
# w_bin.transform = 'r'  # 行标准化

# w_ker = libpysal.weights.DistanceBand.from_dataframe(gdf, threshold=threshold, binary=False)
# w_ker.transform = 'r'  # 行标准化 + 衰减信息保留

# # ===== 3) 打印前几行邻居列表和权重 =====
# def peek_weights(w, name, n_show=5):
#     print(f"\n=== {name} ===")
#     ids = list(w.neighbors.keys())[:n_show]
#     for i in ids:
#         neigh = w.neighbors[i]
#         weights = w.weights[i]
#         print(f"id {i} -> neighbors {neigh} ; weights {np.round(weights, 4)} ; row-sum={sum(weights):.2f}")

# peek_weights(w_bin, "Binary=True (等权, 行标准化后每个邻居权重≈1/度)")
# peek_weights(w_ker, "Binary=False (1/距离衰减, 行标准化后近邻更重)")

# # ===== 4) 可选：把稀疏矩阵导出查看 =====
# def weights_to_edges_df(w):
#     rows = []
#     for i, neighs in w.neighbors.items():
#         for j, wij in zip(neighs, w.weights[i]):
#             rows.append({"i": i, "j": j, "w_ij": wij})
#     return pd.DataFrame(rows)

# edges_bin = weights_to_edges_df(w_bin)
# edges_ker = weights_to_edges_df(w_ker)

# print(edges_bin)
# print(edges_ker)


In [177]:
import os
import geopandas as gpd
import pandas as pd
import numpy as np
import libpysal
from spreg import ML_Error, ML_Lag, OLS
# ML_Error_Het

# 设置路径
grid_folder = r"D:\seoul\grids\lst_map\final_clean\480_based"
output_folder = os.path.join(grid_folder, 'statistics')
os.makedirs(output_folder, exist_ok=True)

results_list = []
def show_formula(model):
    # yongl
    coefs = model.betas.flatten()
    vars_ = model.name_x
    formula = f"{model.name_y[0]} = "
    terms = [f"{coef:.4f}*{var}" for coef, var in zip(coefs, vars_)]
    
    # 添加空间参数
    if hasattr(model, "rho"):
        terms.append(f"{model.rho:.4f}*Wy")
    if hasattr(model, "lambda_"):
        terms.append(f"{model.lambda_:.4f}*error")
    
    formula += " + ".join(terms)
    return formula


for year in [2023, 2016]:
    target_vars = [f'nor_{year}', f'ext_{year}', f'hr_{year}']
    explanatory_vars = ['BCR', 'BHV',  'SVF', 'NDVI', 'EV', 'WR', 'Dist_W', 'Dist_P', 'Dist_M']

    for filename in os.listdir(grid_folder):
        if filename.endswith('.shp') and filename.startswith(f'city{year}_lst_ratio_grid_480m'):
            path = os.path.join(grid_folder, filename)
            gdf = gpd.read_file(path)
            gdf = gdf.replace([np.inf, -np.inf], np.nan)

            for target in target_vars:
                if target not in gdf.columns:
                    continue

                data = gdf[explanatory_vars + [target]].dropna()
                if data.empty:
                    continue

                data_gdf = gdf.loc[data.index]
                print("CRS:", data_gdf.crs)
                # 创建基于距离的空间权重矩阵
                threshold = 1000
                # 设置binary = False, 就可以按照距离衰减
                w = libpysal.weights.DistanceBand.from_dataframe(data_gdf, threshold=threshold, binary=False)
                w.transform = 'r'  # 依然可以行标准化


                yi = np.array(data[target]).reshape(-1, 1)
                x = np.array(data[explanatory_vars])
                n, k = x.shape
                w_name = 'W1000'
                ds_name = f'yr{year}'

                # === 模型估计 ===
                models = {}

                # 1. SLX
                models['SLX'] = OLS(yi, x, w=w, slx_lags=1, spat_diag=True, moran=True,
                 name_w=w_name,name_ds=ds_name)
                
                # 2. SLM
                models['SLM'] = ML_Lag(yi, x, w=w, method="full",
                                       name_w=w_name, name_ds=ds_name,
                                       spat_impacts=['simple','full'])

                # 3. SDM
                models['SDM'] = ML_Lag(yi, x, w=w, slx_lags=1,
                                       name_w=w_name, name_ds=ds_name,
                                       spat_impacts=['simple','full'],
                                       spat_diag=True)

                # 4. SEM
                models['SEM']  = ML_Error(yi, x, w=w, method="full",
                     name_w=w_name,name_ds=ds_name)

                # 5. SDEM
                models['SDEM']  = ML_Error(yi, x, w=w, slx_lags=1,
                    name_w=w_name, name_ds=ds_name)

                # === 提取 AIC, BIC, logLik ===
                for model_name, model in models.items():
                    loglik = model.logll
                    aic = model.aic

                    # 参数数量估算（β + 空间项 + σ²）
                    if model_name in ['SLX']:
                        n_params = k + k + 2  # k原始 + k滞后 + 常数 + σ²
                    elif model_name in ['SEM']:
                        n_params = k + 2      # k自变量 + λ + σ²
                    elif model_name in ['SLM']:
                        n_params = k + 2      # k自变量 + ρ + σ²
                    elif model_name in ['SDM', 'SDEM']:
                        n_params = 2 * k + 3  # k原始 + k滞后 + ρ/λ + 常数 + σ²
                    else:
                        n_params = k + 2

                    bic = -2 * loglik + n_params * np.log(n)

                    results_list.append({
                        "Year": year,
                        "Grid": filename.split("_")[4],
                        "Target": target,
                        "Model": model_name,
                        "AIC": round(aic, 2),
                        "BIC": round(bic, 2),
                        "LogLik": round(loglik, 2),
                        "N": n
                    })
print(show_formula(models['SDM']))
print(show_formula(models['SDEM']))

# === 汇总输出 ===
results_df = pd.DataFrame(results_list)
results_df = results_df.sort_values(["Year", "Target", "Model"])
output_path = os.path.join(output_folder, "SLX_SEM_SLM_SDM_SDEM_AIC_BIC.xlsx")
results_df.to_excel(output_path, index=False)

print(f"✅ 所有模型（SLX/SEM/SLM/SDM/SDEM）AIC/BIC 结果已保存到：\n{output_path}")

CRS: EPSG:5181
('WARNING: ', 1392, ' is an island (no neighbors)')
('WARNING: ', 1571, ' is an island (no neighbors)')
CRS: EPSG:5181
('WARNING: ', 1392, ' is an island (no neighbors)')
('WARNING: ', 1571, ' is an island (no neighbors)')
CRS: EPSG:5181
('WARNING: ', 1392, ' is an island (no neighbors)')
('WARNING: ', 1571, ' is an island (no neighbors)')
CRS: EPSG:5181
('WARNING: ', 1308, ' is an island (no neighbors)')
('WARNING: ', 1377, ' is an island (no neighbors)')
('WARNING: ', 1411, ' is an island (no neighbors)')
('WARNING: ', 1471, ' is an island (no neighbors)')
('WARNING: ', 1525, ' is an island (no neighbors)')
('WARNING: ', 1690, ' is an island (no neighbors)')
CRS: EPSG:5181
('WARNING: ', 1308, ' is an island (no neighbors)')
('WARNING: ', 1377, ' is an island (no neighbors)')
('WARNING: ', 1411, ' is an island (no neighbors)')
('WARNING: ', 1471, ' is an island (no neighbors)')
('WARNING: ', 1525, ' is an island (no neighbors)')
('WARNING: ', 1690, ' is an island (no ne

In [179]:
import os
import geopandas as gpd
import pandas as pd
import numpy as np
import libpysal
from spreg import ML_Error, ML_Lag, OLS
# ML_Error_Het

# 设置路径
grid_folder = r"D:\seoul\grids\lst_map\final_clean\480_based"
output_folder = os.path.join(grid_folder, 'statistics')
os.makedirs(output_folder, exist_ok=True)

results_list = []

for year in [2023, 2016]:
    target_vars = [f'nor_{year}', f'ext_{year}', f'hr_{year}']
    explanatory_vars = ['BCR', 'BHV', 'NDVI', 'SVF', 'EV', 'WR']

    for filename in os.listdir(grid_folder):
        if filename.endswith('.shp') and filename.startswith(f'city{year}_lst_ratio_grid_480m'):
            path = os.path.join(grid_folder, filename)
            gdf = gpd.read_file(path)
            gdf = gdf.replace([np.inf, -np.inf], np.nan)

            for target in target_vars:
                if target not in gdf.columns:
                    continue

                data = gdf[explanatory_vars + [target]].dropna()
                if data.empty:
                    continue

                data_gdf = gdf.loc[data.index]
                print("CRS:", data_gdf.crs)
                # 创建基于距离的空间权重矩阵
                threshold = 1000
                # 设置binary = False, 就可以按照距离衰减
                w = libpysal.weights.DistanceBand.from_dataframe(data_gdf, threshold=threshold, binary=False)
                w.transform = 'r'  # 依然可以行标准化


                yi = np.array(data[target]).reshape(-1, 1)
                x = np.array(data[explanatory_vars])
                n, k = x.shape
                w_name = 'W1000'
                ds_name = f'yr{year}'

                # === 模型估计 ===
                models = {}

                # 1. SLX
                models['SLX'] = OLS(yi, x, w=w, slx_lags=1, spat_diag=True, moran=True,
                 name_w=w_name,name_ds=ds_name)
                
                # 2. SLM
                models['SLM'] = ML_Lag(yi, x, w=w, method="full",
                                       name_w=w_name, name_ds=ds_name,
                                       spat_impacts=['simple','full'])

                # 3. SDM
                models['SDM'] = ML_Lag(yi, x, w=w, slx_lags=1,
                                       name_w=w_name, name_ds=ds_name,
                                       spat_impacts=['simple','full'],
                                       spat_diag=True)

                # 4. SEM
                models['SEM']  = ML_Error(yi, x, w=w, method="full",
                     name_w=w_name,name_ds=ds_name)

                # 5. SDEM
                models['SDEM']  = ML_Error(yi, x, w=w, slx_lags=1,
                    name_w=w_name, name_ds=ds_name)

                # === 提取 AIC, BIC, logLik ===
                for model_name, model in models.items():
                    loglik = model.logll
                    aic = model.aic

                    # 参数数量估算（β + 空间项 + σ²）
                    if model_name in ['SLX']:
                        n_params = k + k + 2  # k原始 + k滞后 + 常数 + σ²
                    elif model_name in ['SEM']:
                        n_params = k + 2      # k自变量 + λ + σ²
                    elif model_name in ['SLM']:
                        n_params = k + 2      # k自变量 + ρ + σ²
                    elif model_name in ['SDM', 'SDEM']:
                        n_params = 2 * k + 3  # k原始 + k滞后 + ρ/λ + 常数 + σ²
                    else:
                        n_params = k + 2

                    bic = -2 * loglik + n_params * np.log(n)

                    results_list.append({
                        "Year": year,
                        "Grid": filename.split("_")[4],
                        "Target": target,
                        "Model": model_name,
                        "AIC": round(aic, 2),
                        "BIC": round(bic, 2),
                        "LogLik": round(loglik, 2),
                        "N": n
                    })

# === 汇总输出 ===
results_df = pd.DataFrame(results_list)
results_df = results_df.sort_values(["Year", "Target", "Model"])
output_path = os.path.join(output_folder, "part_SLX_SEM_SLM_SDM_SDEM_AIC_BIC.xlsx")
results_df.to_excel(output_path, index=False)

print(f"✅ 所有模型（SLX/SEM/SLM/SDM/SDEM）AIC/BIC 结果已保存到：\n{output_path}")


CRS: EPSG:5181
('WARNING: ', 1392, ' is an island (no neighbors)')
('WARNING: ', 1571, ' is an island (no neighbors)')
CRS: EPSG:5181
('WARNING: ', 1392, ' is an island (no neighbors)')
('WARNING: ', 1571, ' is an island (no neighbors)')
CRS: EPSG:5181
('WARNING: ', 1392, ' is an island (no neighbors)')
('WARNING: ', 1571, ' is an island (no neighbors)')
CRS: EPSG:5181
('WARNING: ', 1308, ' is an island (no neighbors)')
('WARNING: ', 1377, ' is an island (no neighbors)')
('WARNING: ', 1411, ' is an island (no neighbors)')
('WARNING: ', 1471, ' is an island (no neighbors)')
('WARNING: ', 1525, ' is an island (no neighbors)')
('WARNING: ', 1690, ' is an island (no neighbors)')
CRS: EPSG:5181
('WARNING: ', 1308, ' is an island (no neighbors)')
('WARNING: ', 1377, ' is an island (no neighbors)')
('WARNING: ', 1411, ' is an island (no neighbors)')
('WARNING: ', 1471, ' is an island (no neighbors)')
('WARNING: ', 1525, ' is an island (no neighbors)')
('WARNING: ', 1690, ' is an island (no ne

## 05 Model Selection Based on Diagnostics
选择合适的空间回归模型
检验结果	推荐模型
仅 LM-Error 显著	SEM（空间误差模型）
仅 LM-Lag 显著	SLM（空间滞后模型）
二者都显著	SDM（空间Durbin模型） 或进行进一步LR检验对比
SDM 是最一般的模型，可以退化为 SLM 或 SEM。
若 SDM 中某些系数不显著，可考虑退化为简化模型。

## AIC BIC 特殊建模

In [ ]:
import os
import geopandas as gpd
import pandas as pd
import numpy as np
import libpysal
from spreg import ML_Error, ML_Lag
from libpysal.weights import lag_spatial

# -----------------------------
# 设置路径
# -----------------------------
grid_folder = r"D:\seoul\grids\lst_map\final_clean\480_based"
output_folder = os.path.join(grid_folder, 'statistics')
os.makedirs(output_folder, exist_ok=True)

# -----------------------------
# 准备变量
# -----------------------------
results_list = []
def show_formula(model, model_type='SDM'):
    # 处理 y 名称
    y_name = model.name_y if isinstance(model.name_y, str) else model.name_y[0]

    coefs = model.betas.flatten()
    vars_ = model.name_x

    terms = []
    for coef, var in zip(coefs, vars_):
        if var.lower() in ['const', 'constant']:  # 常数项
            terms.append(f"{coef:.4f}")
        else:
            terms.append(f"{coef:.4f}*{var}")

    formula = f"{y_name} = "

    # SDM rho 处理
    if model_type == 'SDM' and hasattr(model, 'rho'):
        rho_term = f"{model.rho:.4f}*W{y_name}"
        formula += " + ".join(terms[:1] + [rho_term] + terms[1:])  # 常数项 0 + Wy + X + WX
        return formula

    # SDEM lambda 处理
    if model_type == 'SDEM' and hasattr(model, 'lambda_'):
        terms.append(f"{model.lambda_:.4f}*error")

    formula += " + ".join(terms)
    return formula


param_list = []

for year in [2023, 2016]:
    target_vars = [f'nor_{year}', f'ext_{year}', f'hr_{year}']
    explanatory_vars = ['BCR', 'BHV',  'SVF', 'NDVI', 'EV', 'WR', 'Dist_W', 'Dist_P', 'Dist_M']
    explanatory_vars_clean = ['BCR', 'BHV',  'SVF', 'NDVI', 'EV', 'WR']

    for filename in os.listdir(grid_folder):
        if filename.endswith('.shp') and filename.startswith(f'city{year}_lst_ratio_grid_480m'):
            path = os.path.join(grid_folder, filename)
            gdf = gpd.read_file(path)
            gdf = gdf.replace([np.inf, -np.inf], np.nan)

            for target in target_vars:
                if target not in gdf.columns:
                    continue

                # 只保留完整数据
                data = gdf[explanatory_vars + [target]].dropna()
                if data.empty:
                    continue

                data_gdf = gdf.loc[data.index]
                print(f"CRS: {data_gdf.crs}, File: {filename}, Target: {target}")

                # -----------------------------
                # 空间权重矩阵
                # -----------------------------
                threshold = 1000
                w = libpysal.weights.DistanceBand.from_dataframe(data_gdf, threshold=threshold, binary=False)
                w.transform = 'r'  # 行标准化
                w_name = 'W1000'
                ds_name = f'yr{year}'

                # -----------------------------
                # y 与 X
                # -----------------------------
                yi = data[target].values.reshape(-1, 1)
                X_main = data[explanatory_vars].values
                WX_clean = lag_spatial(w, data[explanatory_vars_clean].values)
                X_all = np.hstack([np.ones((len(data), 1)), X_main, WX_clean])
                name_x = ['const'] + explanatory_vars + [f"W_{v}" for v in explanatory_vars_clean]
                # print(len(name_x), X_all.shape[1])
                
                n, k = X_main.shape  # 原始自变量数量

                # -----------------------------
                # 模型估计
                # -----------------------------
                models = {}

                # SDM
                models['SDM'] = ML_Lag(
                    yi, X_all, w=w,
                    name_y=target, name_x=name_x,
                    name_w=w_name, name_ds=ds_name,
                    spat_diag=True, spat_impacts=['full']
                )

                # SDEM
                models['SDEM'] = ML_Error(
                    yi, X_all, w=w,
                    name_y=target, name_x=name_x,
                    name_w=w_name, name_ds=ds_name,
                    spat_diag=True
                )

                # -----------------------------
                # 提取 AIC/BIC/logLik
                # -----------------------------
                for model_name, model in models.items():
                    loglik = model.logll
                    aic = model.aic

                    # 参数数量估算
                    if model_name in ['SDM', 'SDEM']:
                        n_params = 2 * k + 3  # k原始 + k滞后 + ρ/λ + 常数 + σ²
                    else:
                        n_params = k + 2

                    bic = -2 * loglik + n_params * np.log(n)

                    results_list.append({
                        "Year": year,
                        "Grid": filename.split("_")[4],
                        "Target": target,
                        "Model": model_name,
                        "AIC": round(aic, 2),
                        "BIC": round(bic, 2),
                        "LogLik": round(loglik, 2),
                        "N": n
                    })

                    formula_sdm = show_formula(models['SDM'], model_type='SDM')
                    print(formula_sdm)
                    formula_sdem = show_formula(models['SDEM'], model_type='SDEM')

                    # 系数转成 dict，列名是变量名
                    coef_sdm = dict(zip(models['SDM'].name_x, models['SDM'].betas.flatten()))
                    coef_sdem = dict(zip(models['SDEM'].name_x, models['SDEM'].betas.flatten()))

                    #  lambda_ 加进去, rho 不用加是因为已经有Wy了
                    if hasattr(models['SDEM'], 'lambda_'):
                        coef_sdem['lambda'] = models['SDEM'].lambda_


                # param_list.append({
                #     "Year": year,
                #     "Grid": filename.split("_")[4],
                #     "Target": target,
                #     "Model": "SDM",
                #     "Formula": formula_sdm,
                #     **coef_sdm
                # })

                param_list.append({
                    "Year": year,
                    "Grid": filename.split("_")[4],
                    "Target": target,
                    "Model": "SDEM",
                    "Formula": formula_sdem,
                    **coef_sdem
                })


# -----------------------------
# 汇总输出
# -----------------------------
results_df = pd.DataFrame(results_list)
results_df = results_df.sort_values(["Year", "Target", "Model"])
output_path = os.path.join(output_folder, "SDM_SDEM_clean_AIC_BIC.xlsx")
results_df.to_excel(output_path, index=False)
all_params_df = pd.DataFrame(param_list)
param_output = os.path.join(output_folder, "SDEM_all_params.xlsx")
all_params_df.to_excel(param_output, index=False)


print(f"✅ 所有模型（SDM/SDEM）AIC/BIC 结果已保存到：\n{output_path}")

CRS: EPSG:5181, File: city2023_lst_ratio_grid_480m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp, Target: nor_2023
('WARNING: ', 1392, ' is an island (no neighbors)')
('WARNING: ', 1571, ' is an island (no neighbors)')


C:\Users\owner\AppData\Roaming\Python\Python312\site-packages\scipy\sparse\_data.py:128: RuntimeWarning: divide by zero encountered in reciprocal
  return self._with_data(data ** n)
C:\Users\owner\AppData\Roaming\Python\Python312\site-packages\libpysal\weights\util.py:826: UserWarning: The weights matrix is not fully connected: 
 There are 7 disconnected components.
 There are 2 islands with ids: 1392, 1571.
  w = W(neighbors, weights, ids, **kwargs)
C:\Users\owner\AppData\Roaming\Python\Python312\site-packages\libpysal\weights\distance.py:844: UserWarning: The weights matrix is not fully connected: 
 There are 7 disconnected components.
 There are 2 islands with ids: 1392, 1571.
  W.__init__(
C:\Users\owner\AppData\Roaming\Python\Python312\site-packages\spreg\ml_error.py:184: RuntimeWarning: Method 'bounded' does not support relative tolerance in x; defaulting to absolute tolerance.
  res = minimize_scalar(


nor_2023 = 30.1599 + 0.5751*Wnor_2023 + 0.0448*BCR + 0.0154*BHV + 10.7982*SVF + -26.2233*NDVI + -0.0125*EV + -0.1693*WR + -0.1100*Dist_W + -0.0776*Dist_P + -0.0106*Dist_M + -0.1046*W_BCR + -0.1183*W_BHV + -20.6142*W_SVF + 12.8149*W_NDVI + -0.0005*W_EV + 0.0747*W_WR + 0.5751*W_nor_2023
nor_2023 = 30.1599 + 0.5751*Wnor_2023 + 0.0448*BCR + 0.0154*BHV + 10.7982*SVF + -26.2233*NDVI + -0.0125*EV + -0.1693*WR + -0.1100*Dist_W + -0.0776*Dist_P + -0.0106*Dist_M + -0.1046*W_BCR + -0.1183*W_BHV + -20.6142*W_SVF + 12.8149*W_NDVI + -0.0005*W_EV + 0.0747*W_WR + 0.5751*W_nor_2023
CRS: EPSG:5181, File: city2023_lst_ratio_grid_480m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp, Target: ext_2023
('WARNING: ', 1392, ' is an island (no neighbors)')
('WARNING: ', 1571, ' is an island (no neighbors)')


C:\Users\owner\AppData\Roaming\Python\Python312\site-packages\scipy\sparse\_data.py:128: RuntimeWarning: divide by zero encountered in reciprocal
  return self._with_data(data ** n)
C:\Users\owner\AppData\Roaming\Python\Python312\site-packages\libpysal\weights\util.py:826: UserWarning: The weights matrix is not fully connected: 
 There are 7 disconnected components.
 There are 2 islands with ids: 1392, 1571.
  w = W(neighbors, weights, ids, **kwargs)
C:\Users\owner\AppData\Roaming\Python\Python312\site-packages\libpysal\weights\distance.py:844: UserWarning: The weights matrix is not fully connected: 
 There are 7 disconnected components.
 There are 2 islands with ids: 1392, 1571.
  W.__init__(
C:\Users\owner\AppData\Roaming\Python\Python312\site-packages\spreg\ml_error.py:184: RuntimeWarning: Method 'bounded' does not support relative tolerance in x; defaulting to absolute tolerance.
  res = minimize_scalar(


ext_2023 = 29.3256 + 0.7471*Wext_2023 + 0.0476*BCR + 0.0144*BHV + 12.4025*SVF + -27.3378*NDVI + -0.0127*EV + -0.1678*WR + -0.0464*Dist_W + -0.0227*Dist_P + -0.0636*Dist_M + -0.1205*W_BCR + -0.1437*W_BHV + -28.2722*W_SVF + 18.7731*W_NDVI + 0.0013*W_EV + 0.1100*W_WR + 0.7471*W_ext_2023
ext_2023 = 29.3256 + 0.7471*Wext_2023 + 0.0476*BCR + 0.0144*BHV + 12.4025*SVF + -27.3378*NDVI + -0.0127*EV + -0.1678*WR + -0.0464*Dist_W + -0.0227*Dist_P + -0.0636*Dist_M + -0.1205*W_BCR + -0.1437*W_BHV + -28.2722*W_SVF + 18.7731*W_NDVI + 0.0013*W_EV + 0.1100*W_WR + 0.7471*W_ext_2023
CRS: EPSG:5181, File: city2023_lst_ratio_grid_480m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp, Target: hr_2023
('WARNING: ', 1392, ' is an island (no neighbors)')
('WARNING: ', 1571, ' is an island (no neighbors)')


...

nor_2016 = 23.2855 + 0.5826*Wnor_2016 + 0.0338*BCR + 0.0128*BHV + 10.9005*SVF + -20.0875*NDVI + -0.0123*EV + -0.1102*WR + -0.0086*Dist_W + -0.0056*Dist_P + -0.0228*Dist_M + -0.1100*W_BCR + -0.1021*W_BHV + -16.4776*W_SVF + 2.6954*W_NDVI + 0.0050*W_EV + 0.0332*W_WR + 0.5826*W_nor_2016
nor_2016 = 23.2855 + 0.5826*Wnor_2016 + 0.0338*BCR + 0.0128*BHV + 10.9005*SVF + -20.0875*NDVI + -0.0123*EV + -0.1102*WR + -0.0086*Dist_W + -0.0056*Dist_P + -0.0228*Dist_M + -0.1100*W_BCR + -0.1021*W_BHV + -16.4776*W_SVF + 2.6954*W_NDVI + 0.0050*W_EV + 0.0332*W_WR + 0.5826*W_nor_2016
CRS: EPSG:5181, File: city2016_lst_ratio_grid_480m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp, Target: ext_2016
('WARNING: ', 1308, ' is an island (no neighbors)')
('WARNING: ', 1377, ' is an island (no neighbors)')
('WARNING: ', 1411, ' is an island (no neighbors)')
('WARNING: ', 1471, ' is an island (no neighbors)')
('WARNING: ', 1525, ' is an island (no neighbors)')
('WARNING: ', 1690, ' is an island (no neighbors)')


C:\Users\owner\AppData\Roaming\Python\Python312\site-packages\scipy\sparse\_data.py:128: RuntimeWarning: divide by zero encountered in reciprocal
  return self._with_data(data ** n)
C:\Users\owner\AppData\Roaming\Python\Python312\site-packages\libpysal\weights\util.py:826: UserWarning: The weights matrix is not fully connected: 
 There are 12 disconnected components.
 There are 6 islands with ids: 1308, 1377, 1411, 1471, 1525, 1690.
  w = W(neighbors, weights, ids, **kwargs)
C:\Users\owner\AppData\Roaming\Python\Python312\site-packages\libpysal\weights\distance.py:844: UserWarning: The weights matrix is not fully connected: 
 There are 12 disconnected components.
 There are 6 islands with ids: 1308, 1377, 1411, 1471, 1525, 1690.
  W.__init__(
C:\Users\owner\AppData\Roaming\Python\Python312\site-packages\spreg\ml_error.py:184: RuntimeWarning: Method 'bounded' does not support relative tolerance in x; defaulting to absolute tolerance.
  res = minimize_scalar(


ext_2016 = 33.0126 + 0.5725*Wext_2016 + 0.0520*BCR + 0.0118*BHV + 12.5920*SVF + -37.3964*NDVI + -0.0154*EV + -0.1850*WR + -0.0849*Dist_W + -0.0030*Dist_P + -0.0828*Dist_M + -0.1396*W_BCR + -0.1200*W_BHV + -21.7296*W_SVF + 14.0294*W_NDVI + 0.0047*W_EV + 0.0758*W_WR + 0.5725*W_ext_2016
ext_2016 = 33.0126 + 0.5725*Wext_2016 + 0.0520*BCR + 0.0118*BHV + 12.5920*SVF + -37.3964*NDVI + -0.0154*EV + -0.1850*WR + -0.0849*Dist_W + -0.0030*Dist_P + -0.0828*Dist_M + -0.1396*W_BCR + -0.1200*W_BHV + -21.7296*W_SVF + 14.0294*W_NDVI + 0.0047*W_EV + 0.0758*W_WR + 0.5725*W_ext_2016
CRS: EPSG:5181, File: city2016_lst_ratio_grid_480m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp, Target: hr_2016
('WARNING: ', 1308, ' is an island (no neighbors)')
('WARNING: ', 1377, ' is an island (no neighbors)')
('WARNING: ', 1411, ' is an island (no neighbors)')
('WARNING: ', 1471, ' is an island (no neighbors)')
('WARNING: ', 1525, ' is an island (no neighbors)')
('WARNING: ', 1690, ' is an island (no neighbors)')

C:\Users\owner\AppData\Roaming\Python\Python312\site-packages\scipy\sparse\_data.py:128: RuntimeWarning: divide by zero encountered in reciprocal
  return self._with_data(data ** n)
C:\Users\owner\AppData\Roaming\Python\Python312\site-packages\libpysal\weights\util.py:826: UserWarning: The weights matrix is not fully connected: 
 There are 12 disconnected components.
 There are 6 islands with ids: 1308, 1377, 1411, 1471, 1525, 1690.
  w = W(neighbors, weights, ids, **kwargs)
C:\Users\owner\AppData\Roaming\Python\Python312\site-packages\libpysal\weights\distance.py:844: UserWarning: The weights matrix is not fully connected: 
 There are 12 disconnected components.
 There are 6 islands with ids: 1308, 1377, 1411, 1471, 1525, 1690.
  W.__init__(
C:\Users\owner\AppData\Roaming\Python\Python312\site-packages\spreg\ml_error.py:184: RuntimeWarning: Method 'bounded' does not support relative tolerance in x; defaulting to absolute tolerance.
  res = minimize_scalar(


hr_2016 = -9.1532 + 0.8417*Whr_2016 + -0.0197*BCR + 0.0003*BHV + -1.9617*SVF + 17.0564*NDVI + 0.0032*EV + 0.0734*WR + 0.0407*Dist_W + -0.0050*Dist_P + 0.0409*Dist_M + 0.0547*W_BCR + 0.0456*W_BHV + 8.5564*W_SVF + -13.9486*W_NDVI + 0.0001*W_EV + -0.0581*W_WR + 0.8417*W_hr_2016
hr_2016 = -9.1532 + 0.8417*Whr_2016 + -0.0197*BCR + 0.0003*BHV + -1.9617*SVF + 17.0564*NDVI + 0.0032*EV + 0.0734*WR + 0.0407*Dist_W + -0.0050*Dist_P + 0.0409*Dist_M + 0.0547*W_BCR + 0.0456*W_BHV + 8.5564*W_SVF + -13.9486*W_NDVI + 0.0001*W_EV + -0.0581*W_WR + 0.8417*W_hr_2016
✅ 所有模型（SDM/SDEM）AIC/BIC 结果已保存到：
D:\seoul\grids\lst_map\final_clean\480_based\statistics\SDM_SDEM_clean_AIC_BIC.xlsx


## 06 试试看SDM
检查模型残差分布、异方差性、解释变量的空间效应（直接效应、间接效应）。
分析结果的空间意义与实际城市空间结构/环境变量的关系。

In [ ]:
from libpysal.weights import DistanceBand, lag_spatial
from spreg import ML_Lag
import pandas as pd
import numpy as np
import geopandas as gpd
import os
import re
def star(p):
    return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else '.' if p < 0.1 else ''

grid_folder = r'D:\seoul\grids\lst_map\final_clean\480_based'
output_file = r'D:\seoul\grids\lst_map\final_clean\480_based\statistics\SDM_effects.xlsx'

for year in [2016,2023]:
    target_vars = [f'nor_{year}', f'ext_{year}', f'hr_{year}']
    explanatory_vars = ['BCR', 'BHV',  'SVF', 'NDVI', 'EV', 'WR', 'Dist_P', 'Dist_M', 'Dist_W']
    explanatory_vars_clean = ['BCR', 'BHV',  'SVF', 'NDVI', 'EV', 'WR']
    results_by_target = {t: [] for t in target_vars}
    # 使用 ExcelWriter，mode='a' 可以追加 sheet（如果文件存在）
    with pd.ExcelWriter(output_file, engine='openpyxl', mode='a' if os.path.exists(output_file) else 'w') as writer:

        for filename in os.listdir(grid_folder):
            if filename.endswith('480m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp') and filename.startswith(f'city{year}'):
                path = os.path.join(grid_folder, filename)
                gdf = gpd.read_file(path).replace([np.inf, -np.inf], np.nan)

                for target in target_vars:
                    if target not in gdf.columns:
                        continue

                    data = gdf[explanatory_vars + [target]].dropna()
                    if data.empty:
                        continue

                    data_gdf = gdf.loc[data.index]
                    w = DistanceBand.from_dataframe(data_gdf, threshold=1000, binary=False)
                    w.transform = 'r'

                    # y
                    y = data[target].values.reshape(-1, 1)
                    
                    # X: 原变量
                    X_main = data[explanatory_vars].values

                    # WX_clean: 只 lag clean 变量
                    WX_clean = lag_spatial(w, data[explanatory_vars_clean].values)
                    # print(explanatory_vars)
                    #  # print(explanatory_vars_clean)

                    # 合并常数 + X_main + WX_clean
                    X_all = np.hstack([np.ones((len(data), 1)), X_main, WX_clean])

                    name_x = ['const'] + explanatory_vars + [f"W_{v}" for v in explanatory_vars_clean]

                    # 模型
                    model = ML_Lag(
                        y, X_all, w=w,
                        name_y=target, name_x=name_x,
                        spat_diag=True, spat_impacts=['full'] #或 'simple', 
                    )

                    # ==================================================
                    summary_str = str(model.summary)
                    lines = summary_str.split("\n")

                    impacts_data = []
                    capture = False
                    for line in lines:
                        if "SPATIAL LAG MODEL IMPACTS" in line:
                            capture = True
                            continue
                        if capture:
                            if not line.strip():  # 遇到空行就停
                                break
                            # 跳过分隔线和表头
                            if "Direct" in line and "Indirect" in line and "Total" in line:
                                continue
                            if set(line.strip()) == {"-"}:  # 全是横线
                                continue
                            parts = re.split(r"\s+", line.strip())
                            if len(parts) == 4:
                                var, direct, indirect, total = parts
                                impacts_data.append([var, float(direct), float(indirect), float(total)])

                    df_impacts = pd.DataFrame(impacts_data, columns=["Variable","Direct","Indirect","Total"])
                    print(df_impacts)

                    # 👉 保存 Excel
                    # 保存到 Excel，不同 target 用 sheet 名
                    sheet_name = target
                    df_impacts.to_excel(writer, sheet_name=sheet_name, index=False)
                    print(f"Saved sheet: {sheet_name}")
                    # ==================================================
        #             betas = model.betas.flatten()
        #             stats = model.z_stat

        #             rows = []
        #             for i, var in enumerate(name_x):
        #                 coef = f"{betas[i]:.4f}{star(stats[i][1])}"
        #                 tval = round(stats[i][0], 3)
        #                 pval = round(stats[i][1], 4)
        #                 rows.append([var, coef, tval, pval])

        #             rho_coef = f"{betas[-1]:.4f}{star(stats[-1][1])}"
        #             rho_t = round(stats[-1][0], 3)
        #             rho_p = round(stats[-1][1], 4)
        #             rows.append(["rho", rho_coef, rho_t, rho_p])

        #             rows.append(["Log-likelihood", round(float(model.logll), 4), "", ""])
        #             r2_val = getattr(model, 'pr2', getattr(model, 'r2', np.nan))
        #             rows.append(["R²", round(float(r2_val), 4), "", ""])
        #             rows.append(["", "", "", ""])

        #             df_file = pd.DataFrame(rows, columns=["Feature", "Coefficient", "t-value", "p-value"])
        #             results_by_target[target].append(df_file)

        # outdir = os.path.join(grid_folder, 'statistics')
        # os.makedirs(outdir, exist_ok=True)
        # out_xlsx = os.path.join(outdir, f'{year}_SDM.xlsx')

        # with pd.ExcelWriter(out_xlsx) as wtr:
        #     for target in target_vars:
        #         if not results_by_target[target]:
        #             continue
        #         df_target = pd.concat(results_by_target[target], ignore_index=True)
        #         df_target.to_excel(wtr, sheet_name=target, index=False)

        # print(f"✅ 已生成：{out_xlsx}")


C:\Users\owner\AppData\Roaming\Python\Python312\site-packages\scipy\sparse\_data.py:128: RuntimeWarning: divide by zero encountered in reciprocal
  return self._with_data(data ** n)
C:\Users\owner\AppData\Roaming\Python\Python312\site-packages\libpysal\weights\util.py:826: UserWarning: The weights matrix is not fully connected: 
 There are 12 disconnected components.
 There are 6 islands with ids: 1308, 1377, 1411, 1471, 1525, 1690.
  w = W(neighbors, weights, ids, **kwargs)
C:\Users\owner\AppData\Roaming\Python\Python312\site-packages\libpysal\weights\distance.py:844: UserWarning: The weights matrix is not fully connected: 
 There are 12 disconnected components.
 There are 6 islands with ids: 1308, 1377, 1411, 1471, 1525, 1690.
  W.__init__(


('WARNING: ', 1308, ' is an island (no neighbors)')
('WARNING: ', 1377, ' is an island (no neighbors)')
('WARNING: ', 1411, ' is an island (no neighbors)')
('WARNING: ', 1471, ' is an island (no neighbors)')
('WARNING: ', 1525, ' is an island (no neighbors)')
('WARNING: ', 1690, ' is an island (no neighbors)')


NameError: name 're' is not defined

## 06 试试看SDEM

In [ ]:
from libpysal.weights import DistanceBand, lag_spatial
from spreg import ML_Lag
import pandas as pd
import numpy as np
import geopandas as gpd
import os

def star(p):
    return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else '.' if p < 0.1 else ''

grid_folder = r'D:\seoul\grids\lst_map\final_clean\480_based'
output_fig_dir = r'D:\seoul\grids\lst_map\final_clean\480_based\statistics\fig\figures'
output_file = r'D:\seoul\grids\lst_map\final_clean\480_based\statistics\SDEM_effects.xlsx'

for year in [2016, 2023]:
    target_vars = [f'nor_{year}', f'ext_{year}', f'hr_{year}']
    explanatory_vars = ['BCR', 'BHV',  'SVF', 'NDVI', 'EV', 'WR', 'Dist_P', 'Dist_M', 'Dist_W']
    explanatory_vars_clean = ['BCR', 'BHV',  'SVF', 'NDVI', 'EV', 'WR']
    results_by_target = {t: [] for t in target_vars}
    # 使用 ExcelWriter，mode='a' 可以追加 sheet（如果文件存在）
    with pd.ExcelWriter(output_file, engine='openpyxl', mode='a' if os.path.exists(output_file) else 'w') as writer:

        for filename in os.listdir(grid_folder):
            if filename.endswith('480m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp') and filename.startswith(f'city{year}'):
                path = os.path.join(grid_folder, filename)
                gdf = gpd.read_file(path).replace([np.inf, -np.inf], np.nan)

                for target in target_vars:
                    if target not in gdf.columns:
                        continue

                    data = gdf[explanatory_vars + [target]].dropna()
                    if data.empty:
                        continue

                    data_gdf = gdf.loc[data.index]
                    w = DistanceBand.from_dataframe(data_gdf, threshold=1000, binary=False)
                    w.transform = 'r'

                    # y
                    y = data[target].values.reshape(-1, 1)

                    # X: 原变量
                    X_main = data[explanatory_vars].values
                    # print(explanatory_vars)

                    # WX_clean: 只 lag clean 变量
                    WX_clean = lag_spatial(w, data[explanatory_vars_clean].values)
                    # print(explanatory_vars_clean)

                    # 合并常数 + X_main + WX_clean
                    X_all = np.hstack([np.ones((len(data), 1)), X_main, WX_clean])

                    name_x = ['const'] + explanatory_vars + [f"W_{v}" for v in explanatory_vars_clean]

                    # 模型
                    model = ML_Error(
                        y, X_all,
                        w=w,
                        name_y=target,
                        name_x=name_x,
                        spat_diag=True
                    )

                    betas = model.betas.flatten()

                    # ==================================================
                    # 直接效应：原变量系数
                    direct = betas[1:1+len(explanatory_vars)]
                    # 间接效应：WX 系数
                    indirect = betas[1+len(explanatory_vars):]

                    total = []
                    for var in explanatory_vars:
                        if var in explanatory_vars_clean:
                            total.append(direct[explanatory_vars.index(var)] + 
                                        indirect[explanatory_vars_clean.index(var)])
                        else:
                            total.append(direct[explanatory_vars.index(var)])  # 没有 WX，对应间接效应为 0

                    # 构建表格
                    effects_df = pd.DataFrame({
                        "Variable": explanatory_vars,
                        "Direct": direct,
                        "Indirect": [indirect[explanatory_vars_clean.index(v)] if v in explanatory_vars_clean else 0 for v in explanatory_vars],
                        "Total": total
                    })

                    print(f"=== {target} ===")
                    print(effects_df)

                    # 保存到 Excel，不同 target 用 sheet 名
                    sheet_name = target
                    effects_df.to_excel(writer, sheet_name=sheet_name, index=False)
                    print(f"Saved sheet: {sheet_name}")
                    # ==================================================

                    # ====== 绘制曲线 ======
                    for var in explanatory_vars:
                        var_idx = explanatory_vars.index(var)
                        x_vals = np.linspace(data[var].min(), data[var].max(), 50)
                        y_direct = effects_df.loc[var_idx, "Direct"] * x_vals
                        y_indirect = effects_df.loc[var_idx, "Indirect"] * x_vals
                        y_total = effects_df.loc[var_idx, "Total"] * x_vals

                        plt.figure(figsize=(6,4))
                        plt.plot(x_vals, y_direct, label="Direct", color="blue")
                        plt.plot(x_vals, y_indirect, label="Indirect", color="orange")
                        plt.plot(x_vals, y_total, label="Total", color="green")
                        plt.xlabel(var)
                        plt.ylabel(f"Predicted change in {target}")
                        plt.title(f"{target} - Effect of {var}")
                        plt.legend()
                        plt.tight_layout()
                        plt.savefig(os.path.join(output_fig_dir, f"{target}_{var}_effect_curve.png"), dpi=300)
                        plt.close()

                    # ====== 绘制曲线 ======
                    stats = model.z_stat

                    rows = []
                    for i, var in enumerate(name_x):
                        coef = f"{betas[i]:.4f}{star(stats[i][1])}"
                        tval = round(stats[i][0], 3)
                        pval = round(stats[i][1], 4)
                        rows.append([var, coef, tval, pval])

                    rho_coef = f"{betas[-1]:.4f}{star(stats[-1][1])}"
                    rho_t = round(stats[-1][0], 3)
                    rho_p = round(stats[-1][1], 4)
                    rows.append(["rho", rho_coef, rho_t, rho_p])

                    rows.append(["Log-likelihood", round(float(model.logll), 4), "", ""])
                    r2_val = getattr(model, 'pr2', getattr(model, 'r2', np.nan))
                    rows.append(["R²", round(float(r2_val), 4), "", ""])
                    rows.append(["", "", "", ""])

                    df_file = pd.DataFrame(rows, columns=["Feature", "Coefficient", "t-value", "p-value"])
                    results_by_target[target].append(df_file)

            outdir = os.path.join(grid_folder, 'statistics')
            os.makedirs(outdir, exist_ok=True)
            out_xlsx = os.path.join(outdir, f'{year}_SDEM.xlsx')

        with pd.ExcelWriter(out_xlsx) as wtr:
            for target in target_vars:
                if not results_by_target[target]:
                    continue
                df_target = pd.concat(results_by_target[target], ignore_index=True)
                df_target.to_excel(wtr, sheet_name=target, index=False)

        print(f"✅ 已生成：{out_xlsx}")

: 

<!-- ### 绘制图像，考察impact的影响 -->

<!-- ##### 全部的一起看 -->

In [60]:
# import pandas as pd
# import matplotlib.pyplot as plt
# import os

# # 文件路径
# excel_sdm = r'D:\seoul\grids\lst_map\final_clean\480_based\statistics\SDM_RESULT_EFFECT.xlsx'
# excel_sdem = r'D:\seoul\grids\lst_map\final_clean\480_based\statistics\SDEM_RESULT_EFFECT.xlsx'

# output_dir = r'D:\seoul\grids\lst_map\final_clean\480_based\statistics\fig\impact'
# os.makedirs(output_dir, exist_ok=True)

# def plot_impacts(excel_file, prefix):
#     # 读取所有 sheet
#     xls = pd.ExcelFile(excel_file)
#     for sheet in xls.sheet_names:
#         df = pd.read_excel(excel_file, sheet_name=sheet)

#         # 假设 df 结构：
#         # Variable | Direct | Indirect | Total
#         # 如果列名不同，需要改一下
#         if not set(['Direct', 'Indirect', 'Total']).issubset(df.columns):
#             print(f"⚠️ Skip {sheet}, missing columns")
#             continue

#         variables = df['Variable']
#         direct = df['Direct']
#         indirect = df['Indirect']
#         total = df['Total']

#         # 绘图
#         plt.figure(figsize=(10, 6))
#         bar_width = 0.25
#         x = range(len(variables))

#         plt.bar([i - bar_width for i in x], direct, width=bar_width, label='Direct')
#         plt.bar(x, indirect, width=bar_width, label='Indirect')
#         plt.bar([i + bar_width for i in x], total, width=bar_width, label='Total')

#         plt.xticks(x, variables, rotation=45, ha='right')
#         plt.ylabel('Effect Value')
#         plt.title(f'{prefix} - {sheet} Effects')
#         plt.legend()
#         plt.tight_layout()

#         # 保存
#         out_path = os.path.join(output_dir, f"{prefix}_{sheet}_effects.png")
#         plt.savefig(out_path, dpi=300)
#         plt.close()
#         print(f"✅ Saved {out_path}")

# # 运行
# plot_impacts(excel_sdm, "SDM")
# plot_impacts(excel_sdem, "SDEM")


✅ Saved D:\seoul\grids\lst_map\final_clean\480_based\statistics\fig\impact\SDM_nor_2016_effects.png
✅ Saved D:\seoul\grids\lst_map\final_clean\480_based\statistics\fig\impact\SDM_ext_2016_effects.png
✅ Saved D:\seoul\grids\lst_map\final_clean\480_based\statistics\fig\impact\SDM_hr_2016_effects.png
✅ Saved D:\seoul\grids\lst_map\final_clean\480_based\statistics\fig\impact\SDM_nor_2023_effects.png
✅ Saved D:\seoul\grids\lst_map\final_clean\480_based\statistics\fig\impact\SDM_ext_2023_effects.png
✅ Saved D:\seoul\grids\lst_map\final_clean\480_based\statistics\fig\impact\SDM_hr_2023_effects.png
✅ Saved D:\seoul\grids\lst_map\final_clean\480_based\statistics\fig\impact\SDEM_nor_2016_effects.png
✅ Saved D:\seoul\grids\lst_map\final_clean\480_based\statistics\fig\impact\SDEM_ext_2016_effects.png
✅ Saved D:\seoul\grids\lst_map\final_clean\480_based\statistics\fig\impact\SDEM_hr_2016_effects.png
✅ Saved D:\seoul\grids\lst_map\final_clean\480_based\statistics\fig\impact\SDEM_nor_2023_effects.png

<!-- ##### 各自分组看 -->

In [ ]:
# import pandas as pd
# import matplotlib.pyplot as plt
# import os

# # 文件路径
# excel_sdm = r'D:\seoul\grids\lst_map\final_clean\480_based\statistics\SDM_RESULT_EFFECT.xlsx'
# excel_sdem = r'D:\seoul\grids\lst_map\final_clean\480_based\statistics\SDEM_RESULT_EFFECT.xlsx'

# output_dir = r'D:\seoul\grids\lst_map\final_clean\480_based\statistics\fig\impact'
# os.makedirs(output_dir, exist_ok=True)

# def plot_impacts_grouped(excel_file, prefix):
#     xls = pd.ExcelFile(excel_file)
#     for sheet in xls.sheet_names:
#         df = pd.read_excel(excel_file, sheet_name=sheet)

#         if not set(['Variable', 'Direct', 'Indirect', 'Total']).issubset(df.columns):
#             print(f"⚠️ Skip {sheet}, missing columns")
#             continue

#         # SVF/NDVI + W_SVF/W_NDVI 单独一组
#         svf_ndvi_vars = ['NDVI', 'SVF', 'W_NDVI', 'W_SVF']
#         mask_svf_ndvi = df['Variable'].isin(svf_ndvi_vars)
#         mask_others = ~mask_svf_ndvi

#         groups = {'SVF_NDVI': mask_svf_ndvi, 'Others': mask_others}

#         for grp_name, mask in groups.items():
#             sub_df = df[mask]
#             variables = sub_df['Variable']
#             direct = sub_df['Direct']
#             indirect = sub_df['Indirect']
#             total = sub_df['Total']

#             plt.figure(figsize=(10, 6))
#             bar_width = 0.25
#             x = range(len(variables))

#             plt.bar([i - bar_width for i in x], direct, width=bar_width, label='Direct')
#             plt.bar(x, indirect, width=bar_width, label='Indirect')
#             plt.bar([i + bar_width for i in x], total, width=bar_width, label='Total')

#             plt.xticks(x, variables, rotation=45, ha='right')
#             plt.ylabel('Effect Value')
#             plt.title(f'{prefix} - {sheet} Effects ({grp_name})')
#             plt.legend()
#             plt.tight_layout()

#             out_path = os.path.join(output_dir, f"{prefix}_{sheet}_effects_{grp_name}.png")
#             plt.savefig(out_path, dpi=300)
#             plt.close()
#             print(f"✅ Saved {out_path}")

# # 运行
# plot_impacts_grouped(excel_sdm, "SDM")
# plot_impacts_grouped(excel_sdem, "SDEM")


✅ Saved D:\seoul\grids\lst_map\final_clean\480_based\statistics\fig\impact\SDM_nor_2016_effects_SVF_NDVI.png
✅ Saved D:\seoul\grids\lst_map\final_clean\480_based\statistics\fig\impact\SDM_nor_2016_effects_Others.png
✅ Saved D:\seoul\grids\lst_map\final_clean\480_based\statistics\fig\impact\SDM_ext_2016_effects_SVF_NDVI.png
✅ Saved D:\seoul\grids\lst_map\final_clean\480_based\statistics\fig\impact\SDM_ext_2016_effects_Others.png
✅ Saved D:\seoul\grids\lst_map\final_clean\480_based\statistics\fig\impact\SDM_hr_2016_effects_SVF_NDVI.png
✅ Saved D:\seoul\grids\lst_map\final_clean\480_based\statistics\fig\impact\SDM_hr_2016_effects_Others.png
✅ Saved D:\seoul\grids\lst_map\final_clean\480_based\statistics\fig\impact\SDM_nor_2023_effects_SVF_NDVI.png
✅ Saved D:\seoul\grids\lst_map\final_clean\480_based\statistics\fig\impact\SDM_nor_2023_effects_Others.png
✅ Saved D:\seoul\grids\lst_map\final_clean\480_based\statistics\fig\impact\SDM_ext_2023_effects_SVF_NDVI.png
✅ Saved D:\seoul\grids\lst_ma

### 最终SDEM 模型预测

In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt
# from libpysal.weights import DistanceBand, lag_spatial
# from spreg import ML_Error
# import geopandas as gpd
# import os

# # ------------------- 基本参数 -------------------
# grid_folder = r'D:\seoul\grids\lst_map\final_clean\480_based'
# output_fig_dir = os.path.join(grid_folder, 'statistics', 'fig', 'SDEM_PDP')
# os.makedirs(output_fig_dir, exist_ok=True)

# years = [2016, 2023]
# target_vars_template = ['nor_{}', 'ext_{}', 'hr_{}']
# explanatory_vars = ['BCR', 'BHV',  'SVF', 'NDVI', 'EV', 'WR', 'Dist_W', 'Dist_P', 'Dist_M']    # 顺序很讲究
# explanatory_vars_clean = ['BCR', 'BHV',  'SVF', 'NDVI', 'EV', 'WR']    # 顺序很讲究



# def compute_sdem_full(gdf, target, feature, n_points=100, radius=1000, debug=False):

#     # 1) 取参与建模的列并清理 NaN
#     cols = explanatory_vars + [target]
#     data = gdf[cols].dropna()
#     used_idx = data.index
#     data_gdf = gdf.loc[used_idx]

#     # 2) 构造空间权重（顺序与 data 完全一致）
#     w = DistanceBand.from_dataframe(
#         data_gdf, threshold=radius, binary=False, ids=list(used_idx)
#     )
#     w.transform = 'r'

#     # 3) 构造自变量矩阵 [1, X, WX]
#     X_main = data[explanatory_vars].to_numpy()
#     WX = lag_spatial(w, X_main)                         # (N, p)
#     X_all = np.hstack([np.ones((len(data), 1)), X_main, WX])  # (N, 1+2p)
#     name_x = ['const'] + explanatory_vars + [f"W_{v}" for v in explanatory_vars]

#     # 4) 目标变量
#     y = data[target].to_numpy().reshape(-1, 1)

#     # 5) 拟合 SDEM（Durbin-Error：把 WX 拼到 X 里）
#     model = ML_Error(y, X_all, w=w, name_y=target, name_x=name_x, spat_diag=True)
    
#     # 6) 单变量趋势曲线（不含 λ）
#     if feature not in explanatory_vars:
#         print(f"Skip {feature}, not in explanatory_vars")
#         return None, None, None, None
    
#     p = len(explanatory_vars)
#     betas = model.betas.flatten()[:-1]      # 去掉最后一个 λ
#     f_idx = explanatory_vars.index(feature)
#     X_min, X_max = data[feature].min(), data[feature].max()
#     X_grid = np.linspace(X_min, X_max, n_points)

#     # 设计矩阵：保持其他变量=均值，WX=样本 WX 的列均值
#     X_base = np.zeros((n_points, 1 + 2 * p - 3 ))
#     X_base[:, 0] = 1.0
#     X_mean = X_main.mean(axis=0)           # (p,)
#     WX_mean = WX.mean(axis=0)              # (p,)
#     X_base[:, 1:1+p] = X_mean
#     X_base[:, 1 + f_idx] = X_grid   # ???????????
#     X_base[:, 1+p:] = WX_mean

#     y_trend = (X_base @ betas).ravel()     # (n_points,)

#     return X_grid, y_trend, used_idx




# ########################################################################
# #################>##################################
# ########################################################################

# # ------------------- 循环年份、目标变量、特征 -------------------
# # for year in years:
# #     target_vars = [t.format(year) for t in target_vars_template]

# #     # 找 shapefile
# #     for filename in os.listdir(grid_folder):
# #         if filename.endswith('480m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp') and filename.startswith(f'city{year}'):
# #             path = os.path.join(grid_folder, filename)
# #             gdf = gpd.read_file(path).replace([np.inf, -np.inf], np.nan)

# #             for target in target_vars:
# #                 for feature in explanatory_vars_clean:
# #                     X_grid, y_pred = compute_sdem(gdf, target, feature)
# #                     if X_grid is None:
# #                         continue

# #                     # ------------------- 绘图 -------------------
# #                     plt.figure(figsize=(12,8))
# #                     plt.plot(X_grid, y_pred, 'k--', label='SDEM slope')
# #                     plt.xlabel(feature)
# #                     plt.ylabel(target)
# #                     plt.title(f'{feature} effect on {target})')
# #                     plt.grid(True)
# #                     plt.legend()
# #                     plt.tight_layout()

# #                     fig_name = f"SDEM_{year}_{target}_{feature}.png"
# #                     fig_path = os.path.join(output_fig_dir, fig_name)
# #                     plt.savefig(fig_path, dpi=300)
# #                     plt.close()
# #                     print(f"✅ Saved {fig_path}")

## 06 SDEM 模型评估
linearity, normality, homoscedasticity → 是传统回归模型的残差诊断。
residual spatial autocorrelation → 是空间计量模型特有的诊断，要检查残差是否仍然有空间聚集。Moran I 

In [ ]:
from libpysal.weights import DistanceBand, lag_spatial
from spreg import ML_Lag, ML_Error
import pandas as pd
import numpy as np
import geopandas as gpd
import os
import scipy.stats as stats
import matplotlib.pyplot as plt
from statsmodels.stats.diagnostic import het_breuschpagan
import statsmodels.api as sm
from esda.moran import Moran
import warnings
import sys
import seaborn as sns

# 1. 关闭所有 RuntimeWarning 和 UserWarning
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# # 2. 临时屏蔽 stdout（针对 libpysal 直接 print 的 “island warning”）
class DummyFile(object):
    def write(self, x): pass
    def flush(self): pass


grid_folder = r'D:\seoul\grids\lst_map\final_clean\480_based'

for year in [2016, 2023]:
    target_vars = [f'nor_{year}', f'ext_{year}', f'hr_{year}']
    explanatory_vars = ['BCR(%)', 'BHV', 'NDVI', 'SVF', 'EV(m)',
                        'Dist_BP', 'Dist_MT', 'Dist_WB', 'WR(%)']
    explanatory_vars_clean = ['BCR(%)', 'BHV', 'NDVI', 'SVF', 'EV(m)', 'WR(%)']

    results_by_target = {t: [] for t in target_vars}

    for filename in os.listdir(grid_folder):
        if filename.endswith('480m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp') and filename.startswith(f'city{year}'):
            path = os.path.join(grid_folder, filename)
            gdf = gpd.read_file(path).replace([np.inf, -np.inf], np.nan)

            for target in target_vars:
                if target not in gdf.columns:
                    continue

                data = gdf[explanatory_vars + [target]].dropna()
                if data.empty:
                    continue

                data_gdf = gdf.loc[data.index]
                w = DistanceBand.from_dataframe(data_gdf, threshold=1000, binary=False)
                w.transform = 'r'

                # y
                y = data[target].values.reshape(-1, 1)

                # X: 原变量
                X_main = data[explanatory_vars].values
                # print(explanatory_vars)

                # WX_clean: 只 lag clean 变量
                WX_clean = lag_spatial(w, data[explanatory_vars_clean].values)
                # print(explanatory_vars_clean)

                # 合并常数 + X_main + WX_clean
                X_all = np.hstack([np.ones((len(data), 1)), X_main, WX_clean])

                name_x = ['const'] + explanatory_vars + [f"W_{v}" for v in explanatory_vars_clean]

                # 模型
                model = ML_Error(
                    y, X_all,
                    w=w,
                    name_y=target,
                    name_x=name_x,
                    spat_diag=True
                )
                residuals = model.u  # spreg 里残差是 model.u
                fitted = model.predy.flatten()
                plt.scatter(fitted, residuals, alpha=0.5)
                plt.axhline(0, color='red')
                plt.xlabel("Fitted values")
                plt.ylabel("Residuals")
                plt.title(f"{target} Residuals vs Fitted")
                plt.savefig(rf"D:\seoul\grids\lst_map\final_clean\480_based\statistics\fig\{target} Residuals vs Fitted.png")
                plt.show()
                residuals = model.u.flatten()   # 变成 (n,)

                stats.probplot(residuals, dist="norm", plot=plt)

                plt.title(f"{target} Probability plot")
                plt.savefig(rf"D:\seoul\grids\lst_map\final_clean\480_based\statistics\fig\{target} Probability plot.png")
                plt.show()


                residuals = model.u  # spreg 里残差是 model.u
                residuals = model.u.flatten()   # 变成 (n,)
                shapiro_res = stats.shapiro(residuals)
                print(f"{target} Shapiro-Wilk: stat {shapiro_res[0]:.4f}, p-value {shapiro_res[1]:.4f}")

                test = het_breuschpagan(residuals, X_all)
                print(f"{target} Breusch-Pagan test: F-stat {test[2]:.4f}, p-value {test[3]:.4f}")
                    
                mi = Moran(residuals, w)
                print(f"Moran's I:{mi.I:.4f}", f"p-value: {mi.p_sim:.4f}")

                # 绘制直方图和正态分布曲线
                plt.figure(figsize=(8,5))
                sns.histplot(residuals, bins=30, kde=False, color='skyblue', stat='density', label='Residuals')

                # 生成正态曲线
                mu, sigma = np.mean(residuals), np.std(residuals)
                x = np.linspace(min(residuals), max(residuals), 100)
                plt.plot(x, stats.norm.pdf(x, mu, sigma), color='red', lw=2, label='Normal Fit')

                plt.title('Residuals Distribution vs Normal Distribution')
                plt.xlabel('Residuals')
                plt.ylabel('Density')
                plt.legend()
                plt.savefig(rf"D:\seoul\grids\lst_map\final_clean\480_based\statistics\fig\{target} Residuals Distribution vs Normal Distribution.png")
                plt.show()
                # --- Box-Tidwell Test ---
                # 只对连续自变量做
                bt_data = data.copy()

                # 要求自变量 > 0，否则 log 无法计算（可以平移处理）
                for col in explanatory_vars:
                    if (bt_data[col] <= 0).any():
                        bt_data[col] = bt_data[col] + abs(bt_data[col].min()) + 1e-6  # 平移

                # 生成 Box-Tidwell 变量
                for col in explanatory_vars:
                    bt_data[f'{col}_ln'] = bt_data[col] * np.log(bt_data[col])

                # OLS 回归：y ~ X + X*ln(X)
                X_bt = sm.add_constant(bt_data[explanatory_vars + [f'{col}_ln' for col in explanatory_vars]])
                y_bt = bt_data[target]

                bt_model = sm.OLS(y_bt, X_bt).fit()
                print(f"\n===== Box-Tidwell Test for {target} =====")
                # print(bt_model.summary())

                # 提取 Box-Tidwell 部分的显著性

                for col in explanatory_vars:
                    pval = bt_model.pvalues[f'{col}_ln']
                    if pval < 0.05:
                        print(f"  {col}: Nonlinear relationship detected (p={pval:.4f})")
                    else:
                        print(f"  {col}: Linear relationship acceptable (p={pval:.4f})")


In [176]:
import pandas as pd
import numpy as np
import geopandas as gpd
import os
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan
from esda.moran import Moran
from libpysal.weights import DistanceBand
import scipy.stats as stats

grid_folder = r'D:\seoul\grids\lst_map\final_clean\480_based'

for year in [2016, 2023]:
    target_vars = [f'nor_{year}', f'ext_{year}', f'hr_{year}']
    explanatory_vars = ['BCR', 'BHV',  'SVF', 'NDVI', 'EV', 'WR', 'Dist_W', 'Dist_P', 'Dist_M']

    for filename in os.listdir(grid_folder):
        if filename.endswith('480m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp') and filename.startswith(f'city{year}'):
            path = os.path.join(grid_folder, filename)
            gdf = gpd.read_file(path).replace([np.inf, -np.inf], np.nan)

            for target in target_vars:
                if target not in gdf.columns:
                    continue

                data = gdf[explanatory_vars + [target]].dropna()
                if data.empty:
                    continue

                # y
                y = data[target]
                X = sm.add_constant(data[explanatory_vars])

                # --------- OLS 拟合 ---------
                model = sm.OLS(y, X).fit()
                print(f"\n===== OLS results for {year} | {target} =====")
                print(model.summary())

                residuals = model.resid
                fitted = model.fittedvalues

                # --- 残差 vs 拟合 ---
                plt.scatter(fitted, residuals, alpha=0.5)
                plt.axhline(0, color='red')
                plt.xlabel("Fitted values")
                plt.ylabel("Residuals")
                plt.title(f"{target} Residuals vs Fitted (OLS)")
                plt.savefig(rf"D:\seoul\grids\lst_map\final_clean\480_based\statistics\fig\{target}_OLS_residuals_vs_fitted.png")
                plt.close()

                # --- QQ 图 ---
                sm.qqplot(residuals, line="s")
                plt.title(f"{target} QQ Plot (OLS)")
                plt.savefig(rf"D:\seoul\grids\lst_map\final_clean\480_based\statistics\fig\{target}_OLS_qqplot.png")
                plt.close()

                # --- Shapiro-Wilk 正态性 ---
                shapiro_res = stats.shapiro(residuals)
                print(f"{target} Shapiro-Wilk: stat {shapiro_res[0]:.4f}, p-value {shapiro_res[1]:.4f}")

                # --- Breusch-Pagan 异方差 ---
                test = het_breuschpagan(residuals, X)
                print(f"{target} Breusch-Pagan: F-stat {test[2]:.4f}, p-value {test[3]:.4f}")

                # --- Moran’s I 检查空间自相关（残差）---
                data_gdf = gdf.loc[data.index]
                w = DistanceBand.from_dataframe(data_gdf, threshold=1000, binary=True)
                w.transform = 'r'
                mi = Moran(residuals.values, w)
                print(f"{target} Moran's I: {mi.I:.4f}, p-value: {mi.p_sim:.4f}")

                # --- 残差分布直方图 vs 正态 ---
                plt.figure(figsize=(8,5))
                sns.histplot(residuals, bins=30, kde=False, color='skyblue', stat='density', label='Residuals')
                mu, sigma = np.mean(residuals), np.std(residuals)
                x = np.linspace(min(residuals), max(residuals), 100)
                plt.plot(x, stats.norm.pdf(x, mu, sigma), color='red', lw=2, label='Normal Fit')
                plt.title(f"{target} Residuals Distribution (OLS)")
                plt.xlabel('Residuals')
                plt.ylabel('Density')
                plt.legend()
                plt.savefig(rf"D:\seoul\grids\lst_map\final_clean\480_based\statistics\fig\{target}_OLS_residuals_distribution.png")
                plt.close()



===== OLS results for 2016 | nor_2016 =====
                            OLS Regression Results                            
Dep. Variable:               nor_2016   R-squared:                       0.854
Model:                            OLS   Adj. R-squared:                  0.853
Method:                 Least Squares   F-statistic:                     1202.
Date:                Sat, 30 Aug 2025   Prob (F-statistic):               0.00
Time:                        01:01:15   Log-Likelihood:                -2560.1
No. Observations:                1861   AIC:                             5140.
Df Residuals:                    1851   BIC:                             5196.
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const  

# 02 使用GBDT模型

In [26]:
from scipy.stats import uniform
from scipy.stats import randint
from scipy.stats import loguniform

rvs = uniform.rvs(loc=0.545, scale=0.413, size=10000)
print(rvs.min(), rvs.max())
rvs = randint.rvs(5, 14, size=10000)
print(rvs.min(), rvs.max())
rvs = loguniform.rvs(0.002, 0.355, size=10000)
print(rvs.min(), rvs.max())

0.545009185338065 0.9579966376023945
5 13
0.0020003260363583744 0.35466141435571313


## 01 GBDT
### 01 首先specify hyperparameters, 根据paper里面对于xgboost的调参参考，得到了GBDT的几个参数建议
《Comparing hyperparameter tuning methods in machine learning based urban building energy modeling: A study in Chicago - ScienceDirect》

In [ ]:
# 这个代码干的事情其实就是用梯度提升回归树 (GradientBoostingRegressor, GBDT) 对地理网格数据建模，并做特征重要性分析 + 偏依赖图 (PDP) 提取。
import geopandas as gpd
import pandas as pd
import numpy as np
import os
import re
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import RandomizedSearchCV
from sklearn.inspection import PartialDependenceDisplay
from sklearn.model_selection import train_test_split
from sklearn.model_selection import RepeatedKFold
from scipy.stats import randint, uniform, loguniform

# 文件夹路径
grid_folder = r'D:\seoul\grids\lst_map\final_clean\480_based'
def run_gbdt(grid_folder, years=[2016, 2023], n=0):
    for year in years:
        target_vars = [f'nor_{year}', f'ext_{year}', f'hr_{year}']
        explanatory_vars = ['BCR', 'BHV',  'SVF', 'NDVI', 'EV', 'WR', 'Dist_W', 'Dist_P', 'Dist_M','X','Y'] # 顺序很讲究
        # 保存结果
        all_results = []
        pdp_records = []
        r2_comparison = []

        param_dist = {
            'n_estimators': [4168], #4168
            'learning_rate': loguniform(0.002, 0.355), #(0.002, 0.355)
            'subsample': uniform(0.545, 0.413), # [0.545,0.958]
            'max_depth' : randint(5, 14), # [5, 13]
            'min_samples_split':[2], #2
            'max_features': uniform(0.335, 0.581), #[0.335,0.916]
            }

        # === 主循环 ===
        for filename in os.listdir(grid_folder):
            if filename.endswith(f'city{year}_lst_ratio_grid_480m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy.shp'):
                input_path = os.path.join(grid_folder, filename)
                match = re.search(r'(\d{3,5})m', filename)
                grid_size = match.group(1)

                gdf = gpd.read_file(input_path)
                gdf_clean = gdf.replace([np.inf, -np.inf], np.nan).dropna(subset=target_vars + explanatory_vars)

                for target in target_vars:
                    X = gdf_clean[explanatory_vars]
                    y = gdf_clean[target]

                    #####################################################################
                    # 用20个 random seeds
                    np.random.seed(0)  # 固定种子以便复现
                    random_seeds = np.random.choice(10000, size=20, replace=False)
                    n = n
                    #####################################################################
                    for r in [random_seeds[n]]:

                        # 数据划分
                        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=r) # 切20次

                        gbdt = GradientBoostingRegressor(random_state=0)
                        
                        cv = RepeatedKFold(n_splits=5, n_repeats=20, random_state=0)

                        search = RandomizedSearchCV(
                            estimator=gbdt,
                            param_distributions=param_dist,
                            n_iter= 200,
                            scoring='r2',
                            cv=cv, # cross validation
                            verbose=3,
                            n_jobs=-1,
                            random_state=0
                        )

                        search.fit(X_train, y_train)
                        folder = os.path.join(grid_folder, r'Machine Learning')
                        os.makedirs(folder, exist_ok=True)

                        # 先保存一份 checkpoint (CSV，追加方式)
                        checkpoint_path = os.path.join(folder, f"{year}_{target}_seed{n}_checkpoint.csv")
                        df_cv = pd.DataFrame(search.cv_results_)
                        if not os.path.exists(checkpoint_path):
                            df_cv.to_csv(checkpoint_path, index=False)
                        else:
                            df_cv.to_csv(checkpoint_path, mode='a', header=False, index=False)
                        print(f"[SAVE] checkpoint -> {checkpoint_path}")

                        # 再保存原本的 Excel（完整結果）
                        cv_path = os.path.join(folder, f"{n}_{r}_GBDT_{target}_cv_results.xlsx")
                        try:
                            df_cv.to_excel(cv_path, index=False)
                            print(f"[SAVE] CV results -> {cv_path}")
                        except Exception as e:
                            print(f"[ERROR] Save CV results failed: {cv_path} | {e}")
                            raise

                        # 使用测试集评估
                        best_model = search.best_estimator_
                        y_train_pred = best_model.predict(X_train)
                        y_test_pred = best_model.predict(X_test)

                        r2_train = best_model.score(X_train, y_train)
                        r2_test = r2_score(y_test, y_test_pred)

                        rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
                        rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))

                        print(f" {filename} | {target} 最佳参数: {search.best_params_} | R²_train={r2_train:.3f} | R²_test={r2_test:.3f}")

                        for var, importance in zip(explanatory_vars, best_model.feature_importances_):
                            all_results.append({
                                'GridSize': grid_size,
                                'Target': target,
                                'Feature': var,
                                'Random seed':r,
                                'FeatureImportance_TrainModel': round(importance, 4),
                                'Train_R2': round(r2_train, 4),
                                'Train_RMSE': round(rmse_train, 4),
                                'Test_R2': round(r2_test, 4),
                                'Test_RMSE': round(rmse_test, 4),
                                **search.best_params_
                            })
                            r2_comparison.append({
                                'GridSize': grid_size,
                                'Target': target,
                                'Random seed':r,
                                'Train_R2': round(r2_train, 4),
                                'Test_R2': round(r2_test, 4),
                                'Train_RMSE': round(rmse_train, 4),
                                'Test_RMSE': round(rmse_test, 4)
                            })
                

                
                #   保存模型训练后的结果
                df_all = pd.DataFrame(all_results)

                df_all.to_excel(os.path.join(folder, f'{year} GBDT_Random_Search_Results_{n}_{r}.xlsx'), index=False)
                df_r2 = pd.DataFrame(r2_comparison)
                df_r2 = df_r2.sort_values(['Target', 'GridSize'])
                df_r2.to_excel(os.path.join(folder, f'{year} R2_Comparison_Train_vs_Test_{n}_{r}.xlsx'), index=False)
                # print("✅ R² train vs test comparison saved.")

run_gbdt(grid_folder, years=[2023], n=0)

: 

# 关于 prediction 和PDP的出图，在文件 GBDT_tailored.ipynb 中。

In [12]:
# 这个代码会读取你之前保存的 训练集 vs 测试集 R² 指标表格，然后把不同网格大小下的 Train/Test R² 曲线画出来。最后，它会为每个目标变量（nor/ext/hr）分别保存一张对比图到指定文件夹。

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# === 文件路径 ===
grid_folder = r'D:\seoul\grids\lst_map\final_clean\480_based\Machine Learning'
for year in [2016,2023]:
    input_path = os.path.join(grid_folder, f'{year} R2_Comparison_Train_vs_Test.xlsx')
    output_dir = os.path.join(grid_folder, 'figures', 'r2_comparison')
    os.makedirs(output_dir, exist_ok=True)

    # === 读取数据 ===
    df = pd.read_excel(input_path)
    df['GridSize'] = df['GridSize'].astype(int)

    # === 绘图：每个 Target 一个对比图 ===
    for target in df['Target'].unique():
        df_sub = df[df['Target'] == target].sort_values('GridSize')

        plt.figure(figsize=(8, 5))
        plt.plot(df_sub['GridSize'], df_sub['Train_R2'], marker='o', label='Train R²')
        plt.plot(df_sub['GridSize'], df_sub['Test_R2'], marker='o', label='Test R²')

        plt.title(f'R² Comparison — {target}')
        plt.xticks(sorted(df['GridSize'].unique()))  # ✅ 强制显示真实 X 值
        plt.xlabel('Grid Size (m)')
        plt.ylabel('R² Score')
        plt.ylim(0, 1.05)
        plt.grid(True)
        plt.legend()
        plt.tight_layout()

        out_path = os.path.join(output_dir, f'{year} R2_Train_vs_Test_{target}.png')
        plt.savefig(out_path, dpi=300)
        plt.close()
        print(f"📈 Saved R² comparison: {out_path}")


FileNotFoundError: [Errno 2] No such file or directory: 'D:\\seoul\\grids\\lst_map\\final_clean\\480_based\\Machine Learning\\GBDT结果_不含X,Y\\2016 R2_Comparison_Train_vs_Test.xlsx'